# Paper 4 — Manuscript figures

This notebook rebuilds the paper's main descriptive results directly from the automated
`citegraph` outputs. It is read-only over `OUT_DIR`: it loads the checkpoint CSVs written
by [paper4_pipeline.ipynb](paper4_pipeline.ipynb) and never calls the extraction model, so
rerunning it is free and refreshes every result against the latest completed stage.

It reads `works.csv` and `citation_graph.csv` (stage 4) and `authors.csv` and
`author_citations.csv` (stage 6). Run the pipeline notebook through author normalization
first; the sections below raise an explicit error if an artifact is missing.

Artwork, captions, and the exported tables are written to `OUT_DIR/figures/`.

## Setup

In [ ]:
import os
from pathlib import Path

import citegraph

# The replication root is the folder holding citegraph_out/ and institutional/;
# every path below hangs off it, so the package runs wherever it is cloned.
ROOT = next(
    p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
    if (p / "citegraph_out").is_dir() and (p / "institutional").is_dir()
)
print(f"citegraph {citegraph.__version__} from {Path(citegraph.__file__).parent}")
print(f"replication root: {ROOT}")

import pandas as pd

OUT_DIR = ROOT / "citegraph_out"

required_artifacts = ["works.csv", "citation_graph.csv", "authors.csv", "author_citations.csv"]
missing_artifacts = [name for name in required_artifacts if not (OUT_DIR / name).is_file()]
if missing_artifacts:
    raise FileNotFoundError(
        f"{OUT_DIR} is missing {', '.join(missing_artifacts)}. "
        "Run `make pipeline` (or restore citegraph_out/ from the repository) first."
    )

print(f"Pipeline output: {OUT_DIR}")

## Figure style

All figures share one style, implemented in `citegraph.plotting` and documented in
`.claude/skills/styling-econ-figures/`: the conventions of top economics journals
(grayscale-first marks, serif type sized for the printed page, no title or notes inside
the plot). Every figure is a single plot saved to its own PDF and PNG, and captions are
unnumbered, so figures can be arranged and numbered freely in the manuscript. Each
caption is rendered below its figure and saved next to the artwork as `<name>.caption.md`.

In [ ]:
# Shared figure style for every replication figure below.
#
# Every mark comes from citegraph.plotting, which is the single place these
# conventions are defined; the companion notebook paper4_institutional_figures.ipynb
# draws from the same module, so the two figure sets share one visual language by
# construction rather than by convention.
#
# The figures follow economics-journal conventions (AER / QJE / Econometrica):
# grayscale-first marks with black edges, serif type sized for the printed page,
# and no title, subtitle, or source text inside the plot. Every figure is a single
# plot saved to its own file so figures can be arranged freely in the manuscript.
# The caption (title, notes, source) is rendered as Markdown directly under each
# figure and saved next to the artwork as <stem>.caption.md.
# Rules and rationale: .claude/skills/styling-econ-figures/ in this repository.
import matplotlib.pyplot as plt

from citegraph.plotting import (
    GRAYS,
    EDGE_COLOR,
    HATCHES,
    NODE_FILL,
    bar,
    barh,
    econ_style,
    figure_size,
    integer_ticks,
    network_axes,
    percent_ticks,
    reference_line,
    save_figure,
    value_labels,
    year_ticks,
)
from citegraph.plotting import show_caption as _show_caption

FIGURE_DIR = OUT_DIR / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

SOURCE_NOTE = "Authors' calculations from citegraph metadata and bibliography extraction over the paper corpus."


def show_caption(caption, *, notes=None, source=SOURCE_NOTE):
    """Render the caption block under a figure, defaulting to this notebook's source."""
    _show_caption(caption, notes=notes, source=source)

## 1. Publication trends

The figure reports annual publication counts for the core corpus, including years with zero publications. Records with missing or implausible years are reported and excluded rather than silently assigned to a year.

In [ ]:
from datetime import datetime

from citegraph import CitationGraph

# Reload the checkpoint so this analysis reflects the latest completed metadata stage.
trend_papers = CitationGraph.from_out_dir(OUT_DIR).core.reset_index()
trend_papers["Year_extracted"] = pd.to_numeric(trend_papers["Year"], errors="coerce")

current_year = datetime.now().year
valid_year = trend_papers["Year_extracted"].between(1900, current_year + 1)
year_issues = trend_papers.loc[
    ~valid_year,
    [column for column in ["id", "Title", "Year", "source_file"] if column in trend_papers],
].copy()

trend_papers = trend_papers.loc[valid_year].copy()
if trend_papers.empty:
    raise ValueError("No plausible publication years were found among core works.")

trend_papers["Year"] = trend_papers["Year_extracted"].astype(int)
min_year = int(trend_papers["Year"].min())
max_year = int(trend_papers["Year"].max())

annual_index = pd.Index(range(min_year, max_year + 1), name="Year")
pubs_per_year = (
    trend_papers.groupby("Year").size()
    .reindex(annual_index, fill_value=0)
    .rename("Publications")
    .reset_index()
)

trend_papers["Period_start"] = (trend_papers["Year"] // 5) * 5
period_index = pd.Index(
    range(
        int(trend_papers["Period_start"].min()),
        int(trend_papers["Period_start"].max()) + 1,
        5,
    ),
    name="Period_start",
)
pubs_per_period = (
    trend_papers.groupby("Period_start").size()
    .reindex(period_index, fill_value=0)
    .rename("Publications")
    .reset_index()
)
pubs_per_period["Period"] = pubs_per_period["Period_start"].map(
    lambda year: f"{year}–{year + 4}"
)

peak_year = pubs_per_year.loc[pubs_per_year["Publications"].idxmax()]
peak_period = pubs_per_period.loc[pubs_per_period["Publications"].idxmax()]

print(f"Corpus records: {len(trend_papers):,} with plausible years; {len(year_issues):,} excluded")
print(f"Coverage: {min_year}–{max_year}")
print(f"Peak year: {int(peak_year['Year'])} ({int(peak_year['Publications'])} publications)")
print(f"Peak five-year period: {peak_period['Period']} ({int(peak_period['Publications'])} publications)")

if not year_issues.empty:
    display(year_issues)

In [ ]:
# Annual publication counts. Zero-count years are kept so the series is not compressed.
with econ_style():
    fig, ax = plt.subplots(figsize=figure_size("text", ratio=0.55))

    bar(ax, pubs_per_year["Year"], pubs_per_year["Publications"], width=0.75)
    ax.set_xlabel("Year")
    ax.set_ylabel("Number of publications")
    year_ticks(ax, min_year, max_year)
    integer_ticks(ax, "y")
    ax.set_ylim(bottom=0)

    pubs_year_caption = "Publications per year"
    pubs_year_notes = (
        f"Annual number of publications in the corpus, {min_year}–{max_year}, including "
        f"years with no publications ({len(trend_papers):,} publications with plausible "
        f"years; {len(year_issues):,} record(s) with missing or implausible years excluded)."
    )
    pubs_year_files = save_figure(
        fig,
        FIGURE_DIR / "publications_per_year",
        caption=pubs_year_caption,
        notes=pubs_year_notes,
        source=SOURCE_NOTE,
    )
    plt.show()

show_caption(pubs_year_caption, notes=pubs_year_notes)
for path in pubs_year_files:
    print(f"Saved: {path}")

## 2. What the corpus covers

Two hand-coded attributes of the source publications: the broad research topic and the
kind of publication. Neither is extracted from the PDF. Both were read off the papers by
a reviewer and recorded in `work_annotations.csv`, which sits beside the pipeline's output
rather than inside `works.csv` — the canonicalization stage regenerates that file, so a
column added to it would not survive the next run.

A journal article takes the topic its journal has in the reviewer's journal-to-topic
table; chapters, working papers and dissertations were coded from the paper. Eight
publications whose source coding recorded a publication type where a topic belongs were
assigned a topic from the paper text; the reasoning for each, and for every topic changed
to follow the journal table, is kept in the annotation file's `Tema_evidence` column, so
the derived part of the variable stays auditable.

In [ ]:
from citegraph import CitationGraph

# Reload the checkpoint so the composition reflects the current annotation file.
annotated_graph = CitationGraph.from_out_dir(OUT_DIR)
if not annotated_graph.has_annotations:
    raise RuntimeError(
        "No work_annotations.csv in the output directory. Run "
        "paper4/annotations/make_work_annotations.py before this section."
    )

source_attributes = annotated_graph.core
n_sources = len(source_attributes)


def composition(column):
    """Publications per category, with each category's share of the corpus.

    `value_counts` drops publications with no value recorded, so the counts
    describe the publications actually coded; the totals printed below say how
    many that is.
    """
    counts = source_attributes[column].value_counts()
    if counts.empty:
        raise ValueError(f"No source publication has a {column} recorded.")
    table = counts.rename("Publications").rename_axis("Category").reset_index()
    table["Share_pct"] = 100 * table["Publications"] / n_sources
    return table


topic_composition = composition("Tema")
type_composition = composition("Publication_Type")

n_topic_coded = int(topic_composition["Publications"].sum())
n_type_coded = int(type_composition["Publications"].sum())
print(f"{n_sources:,} source publications")
print(f"  topic recorded for {n_topic_coded:,} across {len(topic_composition)} topics")
print(f"  publication type recorded for {n_type_coded:,} across "
      f"{len(type_composition)} types")
display(topic_composition)
display(type_composition)

In [ ]:
# Ascending sort puts the largest category at the top of a horizontal bar chart.
# Horizontal because the topic names are long: rotating tick labels is the last
# resort, not the first.
plot_topics = topic_composition.sort_values(
    ["Publications", "Category"], ascending=[True, False]
)
topic_positions = range(len(plot_topics))

with econ_style():
    fig, ax = plt.subplots(figsize=figure_size("text", ratio=0.45))

    barh(ax, topic_positions, plot_topics["Publications"], height=0.7)
    ax.set_yticks(list(topic_positions))
    ax.set_yticklabels(plot_topics["Category"])
    ax.tick_params(axis="y", length=0)

    # A short ranking: the exact count and corpus share at each bar end are informative.
    for y_position, (_, row) in zip(topic_positions, plot_topics.iterrows(), strict=True):
        ax.text(
            row["Publications"] + 0.4,
            y_position,
            f"{int(row['Publications'])} ({row['Share_pct']:.1f}%)",
            va="center",
            ha="left",
            fontsize="small",
        )

    ax.set_xlabel("Number of publications")
    integer_ticks(ax, "x")
    ax.set_xlim(0, plot_topics["Publications"].max() * 1.22)
    ax.margins(y=0.06)

    topic_caption = "Source publications by research topic"
    # Say "N of M" only when they differ; with full coverage "93 out of 93"
    # reads like a mistake.
    topic_coverage = (
        f"Each of the {n_sources:,} source publications is counted once."
        if n_topic_coded == n_sources
        else f"Counts cover the {n_topic_coded:,} of {n_sources:,} source publications "
             "with a topic recorded."
    )
    topic_notes = (
        f"{topic_coverage} Topics are coded by hand, not extracted from the text: a "
        "journal article takes its journal's topic in the reviewer's journal-to-topic "
        "table, and other publications are coded from the paper. Labels give the count and "
        "its share of the corpus. Eight publications whose source coding named a "
        "publication type where a topic belongs were assigned a topic from the paper text."
    )
    topic_files = save_figure(
        fig,
        FIGURE_DIR / "papers_by_topic",
        caption=topic_caption,
        notes=topic_notes,
        source=SOURCE_NOTE,
    )
    plt.show()

show_caption(topic_caption, notes=topic_notes)
for path in topic_files:
    print(f"Saved: {path}")

In [ ]:
# Same form as the topic figure above, so the two read as one pair.
plot_types = type_composition.sort_values(
    ["Publications", "Category"], ascending=[True, False]
)
type_positions = range(len(plot_types))

with econ_style():
    fig, ax = plt.subplots(figsize=figure_size("text", ratio=0.32))

    barh(ax, type_positions, plot_types["Publications"], height=0.7)
    ax.set_yticks(list(type_positions))
    ax.set_yticklabels(plot_types["Category"])
    ax.tick_params(axis="y", length=0)

    for y_position, (_, row) in zip(type_positions, plot_types.iterrows(), strict=True):
        ax.text(
            row["Publications"] + 0.9,
            y_position,
            f"{int(row['Publications'])} ({row['Share_pct']:.1f}%)",
            va="center",
            ha="left",
            fontsize="small",
        )

    ax.set_xlabel("Number of publications")
    integer_ticks(ax, "x")
    ax.set_xlim(0, plot_types["Publications"].max() * 1.22)
    ax.margins(y=0.10)

    type_caption = "Source publications by type of publication"
    type_coverage = (
        f"Each of the {n_sources:,} source publications is counted once."
        if n_type_coded == n_sources
        else f"Counts cover the {n_type_coded:,} of {n_sources:,} source publications "
             "with a type recorded."
    )
    type_notes = (
        f"{type_coverage} Types are coded by hand: 'Paper' covers journal articles, the "
        "rest are book chapters, working papers and doctoral dissertations. Labels give "
        "the count and its share of the corpus."
    )
    type_files = save_figure(
        fig,
        FIGURE_DIR / "papers_by_publication_type",
        caption=type_caption,
        notes=type_notes,
        source=SOURCE_NOTE,
    )
    plt.show()

show_caption(type_caption, notes=type_notes)
for path in type_files:
    print(f"Saved: {path}")

### JEL classification

A third attribute, and the only one of the three that the paper itself can supply: the
JEL codes printed on the title page. `citegraph jel` reads them back from the converted
markdown into `jel_codes.csv`; papers whose PDF prints none take codes from the
`JEL_Codes` column of `work_annotations.csv`, assigned by hand, and `JEL_Source` says
which is which. How many papers print codes at all is itself a finding about the
corpus: working papers, book chapters and dissertations rarely do.

The figure groups codes at the general-category level (the letter: C for methods, D for
microeconomics, Q for environmental and resource economics). A paper with codes in
several categories is counted once in each.


In [ ]:
import textwrap

from citegraph.jel import JEL_GENERAL_CATEGORIES, core_jel_codes

jel_table = core_jel_codes(OUT_DIR)
jel_source_counts = jel_table["JEL_Source"].value_counts()
n_jel_printed = int(jel_source_counts.get("extracted", 0))
n_jel_manual = int(jel_source_counts.get("manual", 0))
n_jel_coded = n_jel_printed + n_jel_manual
if n_jel_coded == 0:
    raise ValueError(
        "No core paper has JEL codes. Run `citegraph jel --out OUT_DIR` and hand-code "
        "the papers it lists in work_annotations.csv."
    )

jel_composition = (
    jel_table["JEL_General"].explode().dropna().value_counts()
    .rename("Publications").rename_axis("Letter").reset_index()
)
jel_composition["Category"] = jel_composition["Letter"].map(JEL_GENERAL_CATEGORIES)
jel_composition["Share_pct"] = 100 * jel_composition["Publications"] / n_jel_coded
jel_composition = jel_composition[["Letter", "Category", "Publications", "Share_pct"]]

print(f"{n_sources:,} source publications")
print(f"  JEL codes printed in the paper: {n_jel_printed:,}")
print(f"  JEL codes assigned by hand:     {n_jel_manual:,}")
print(f"  no JEL codes:                   {n_sources - n_jel_coded:,}")
print(f"  general categories represented: {len(jel_composition)}")
display(jel_composition)
jel_table.to_csv(FIGURE_DIR / "papers_jel_codes.csv")

# Same form as the topic figure: horizontal bars, largest category at the top.
plot_jel = jel_composition.sort_values(
    ["Publications", "Letter"], ascending=[True, False]
)
jel_positions = range(len(plot_jel))
jel_labels = [
    f"{row.Letter}  " + "\n    ".join(textwrap.wrap(row.Category, 48))
    for row in plot_jel.itertuples()
]

with econ_style():
    fig, ax = plt.subplots(figsize=figure_size("text", ratio=0.045 * len(plot_jel) + 0.12))

    barh(ax, jel_positions, plot_jel["Publications"], height=0.7)
    ax.set_yticks(list(jel_positions))
    ax.set_yticklabels(jel_labels)
    ax.tick_params(axis="y", length=0)

    for y_position, (_, row) in zip(jel_positions, plot_jel.iterrows(), strict=True):
        ax.text(
            row["Publications"] + 0.4,
            y_position,
            f"{int(row['Publications'])} ({row['Share_pct']:.1f}%)",
            va="center",
            ha="left",
            fontsize="small",
        )

    ax.set_xlabel("Number of publications")
    integer_ticks(ax, "x")
    ax.set_xlim(0, plot_jel["Publications"].max() * 1.22)
    ax.margins(y=0.06)

    jel_caption = "Source publications by JEL general category"
    # The appendix figure describes the papers that print codes; the rest are
    # excluded, not hand-coded, and the notes say so. Mention hand-assigned
    # codes only if work_annotations.csv ever supplies any.
    by_hand = f" and {n_jel_manual:,} were assigned by hand" if n_jel_manual else ""
    jel_coverage = (
        f"Counts cover the {n_jel_coded:,} of {n_sources:,} source publications with JEL "
        f"codes: {n_jel_printed:,} print them on the title page{by_hand}. The other "
        f"{n_sources - n_jel_coded:,} publications print no JEL codes and are excluded."
        if n_jel_coded < n_sources
        else f"Every one of the {n_sources:,} source publications has JEL codes: "
             f"{n_jel_printed:,} print them on the title page{by_hand}."
    )
    jel_notes = (
        f"{jel_coverage} Codes are grouped by their JEL general category (the first letter); "
        "a publication whose codes span several categories is counted once in each, so "
        "the bars sum to more than the number of publications. Labels give the count and "
        f"its share of the {n_jel_coded:,} publications with codes."
    )
    jel_files = save_figure(
        fig,
        FIGURE_DIR / "papers_by_jel_category",
        caption=jel_caption,
        notes=jel_notes,
        source=SOURCE_NOTE,
    )
    plt.show()

show_caption(jel_caption, notes=jel_notes)
for path in jel_files:
    print(f"Saved: {path}")


## 3. Citation of Elinor Ostrom

A source publication counts as citing Elinor Ostrom when its extracted bibliography contains at least one reference assigned to a canonical Elinor Ostrom author cluster. This graph-based definition avoids unreliable title keyword searches and counts each source publication only once, even when it cites several Ostrom works.

Because abbreviated citations may produce more than one canonical cluster, the code combines all Ostrom clusters identified as “Elinor” or “E.” and prints both the selected identities and cited works for review.

Three figures follow: the number of core publications per year split by whether they cite Ostrom, the share of each year's publications that cite Ostrom, and the five Ostrom works cited by the most core publications.

In [ ]:
from datetime import datetime

from citegraph import CitationGraph

# Reload all graph artifacts so the result reflects the latest completed pipeline stages.
current_year = datetime.now().year
replication_graph = CitationGraph.from_out_dir(OUT_DIR)
if not replication_graph.has_authors:
    raise RuntimeError("Run the author-normalization stage before this subsection.")

ostrom_candidates = replication_graph.find_author("ostrom").copy()
if ostrom_candidates.empty:
    raise ValueError("No canonical author matching 'Ostrom' was found.")

# Precision-first rule for Elinor Ostrom. Multiple matches are retained to absorb split E./Elinor clusters.
given = ostrom_candidates.get("canonical_given", pd.Series(index=ostrom_candidates.index, dtype=str)).fillna("").str.casefold()
initials = ostrom_candidates.get("initials", pd.Series(index=ostrom_candidates.index, dtype=str)).fillna("").str.upper()
surname = ostrom_candidates.get("surname_norm", pd.Series(index=ostrom_candidates.index, dtype=str)).fillna("").str.casefold()
display_name = ostrom_candidates.get("display_name", pd.Series(index=ostrom_candidates.index, dtype=str)).fillna("").str.casefold()

elinor_mask = (
    surname.eq("ostrom")
    & (
        given.isin(["e", "elinor"])
        | initials.eq("E")
        | display_name.str.contains("elinor", regex=False)
    )
)
elinor_ostrom_authors = ostrom_candidates.loc[elinor_mask].copy()
if elinor_ostrom_authors.empty:
    raise ValueError(
        "Ostrom candidates were found, but none could be identified as Elinor/E. Ostrom. "
        "Inspect ostrom_candidates and adjust the identity rule explicitly."
    )

ostrom_context = pd.concat(
    [
        replication_graph.citation_context_for_author(author_id)
        for author_id in elinor_ostrom_authors.index
    ],
    ignore_index=True,
).drop_duplicates(["citing_paper_id", "cited_reference_id"])
ostrom_context = ostrom_context[
    ostrom_context["citing_paper_id"].isin(replication_graph.core.index)
]

source_papers = replication_graph.core.reset_index().drop_duplicates("id").copy()
ostrom_citing_ids = set(ostrom_context["citing_paper_id"].dropna())
source_papers["cites_elinor_ostrom"] = source_papers["id"].isin(ostrom_citing_ids)

n_source_papers = len(source_papers)
n_citing_ostrom = int(source_papers["cites_elinor_ostrom"].sum())
ostrom_share = n_citing_ostrom / n_source_papers if n_source_papers else 0.0

source_papers["Year_numeric"] = pd.to_numeric(source_papers["Year"], errors="coerce")
valid_source_year = source_papers["Year_numeric"].between(1900, current_year + 1)
ostrom_by_period = source_papers.loc[valid_source_year].copy()
ostrom_by_period["Period_start"] = (ostrom_by_period["Year_numeric"].astype(int) // 5) * 5
ostrom_by_period = (
    ostrom_by_period.groupby("Period_start")
    .agg(
        Publications=("id", "nunique"),
        Citing_Ostrom=("cites_elinor_ostrom", "sum"),
    )
    .reset_index()
)
ostrom_by_period["Share"] = ostrom_by_period["Citing_Ostrom"] / ostrom_by_period["Publications"]
ostrom_by_period["Period"] = ostrom_by_period["Period_start"].map(
    lambda year: f"{year}–{year + 4}"
)

ostrom_reference_audit = (
    ostrom_context[[
        "cited_reference_id",
        "cited_reference_title",
        "cited_reference_year",
    ]]
    .drop_duplicates("cited_reference_id")
    .sort_values(["cited_reference_year", "cited_reference_title"], na_position="last")
    .reset_index(drop=True)
)

print(f"Selected Elinor Ostrom cluster(s): {', '.join(elinor_ostrom_authors.index)}")
print(f"Papers citing Elinor Ostrom: {n_citing_ostrom}/{n_source_papers} ({ostrom_share:.1%})")
print(f"Distinct Ostrom works cited: {len(ostrom_reference_audit)}")

display(elinor_ostrom_authors[[
    column for column in ["display_name", "canonical_given", "initials", "openalex_id", "orcid"]
    if column in elinor_ostrom_authors
]])
display(ostrom_reference_audit)

In [ ]:
# Annual aggregation shared by the two per-year Ostrom figures. Years with no
# publications are kept (zero counts, undefined share).
ostrom_years = source_papers.loc[valid_source_year].copy()
ostrom_years["Year"] = ostrom_years["Year_numeric"].astype(int)
ostrom_first_year = int(ostrom_years["Year"].min())
ostrom_last_year = int(ostrom_years["Year"].max())
ostrom_by_year = (
    ostrom_years.groupby("Year")
    .agg(
        Publications=("id", "nunique"),
        Citing_Ostrom=("cites_elinor_ostrom", "sum"),
    )
    .reindex(pd.Index(range(ostrom_first_year, ostrom_last_year + 1), name="Year"), fill_value=0)
    .reset_index()
)
ostrom_by_year["Citing_Ostrom"] = ostrom_by_year["Citing_Ostrom"].astype(int)
ostrom_by_year["Not_citing_Ostrom"] = ostrom_by_year["Publications"] - ostrom_by_year["Citing_Ostrom"]
ostrom_by_year["Share"] = ostrom_by_year["Citing_Ostrom"] / ostrom_by_year["Publications"].where(
    ostrom_by_year["Publications"] > 0
)

# Publications per year, split by whether they cite Ostrom. Dark = cites, light = does not.
with econ_style():
    fig, ax = plt.subplots(figsize=figure_size("text", ratio=0.55))

    bar(
        ax,
        ostrom_by_year["Year"],
        ostrom_by_year["Citing_Ostrom"],
        width=0.75,
        color=GRAYS[0],
        label="Cites Elinor Ostrom",
    )
    bar(
        ax,
        ostrom_by_year["Year"],
        ostrom_by_year["Not_citing_Ostrom"],
        bottom=ostrom_by_year["Citing_Ostrom"],
        width=0.75,
        color=GRAYS[3],
        label="Does not cite",
    )
    ax.set_xlabel("Year")
    ax.set_ylabel("Number of publications")
    year_ticks(ax, ostrom_first_year, ostrom_last_year)
    integer_ticks(ax, "y")
    ax.set_ylim(0, ostrom_by_year["Publications"].max() * 1.12)
    ax.legend(loc="upper left")

    ostrom_counts_year_caption = "Publications citing Elinor Ostrom, per year"
    ostrom_counts_year_notes = (
        "Annual number of publications, split by whether the publication cites Elinor "
        "Ostrom. A publication counts as citing Ostrom when its extracted bibliography "
        "contains at least one reference matched to a canonical Elinor Ostrom author "
        "cluster; each publication counts once regardless of how many Ostrom works it "
        f"cites. Overall, {n_citing_ostrom} of {n_source_papers} publications "
        f"({ostrom_share:.1%}) cite Ostrom."
    )
    ostrom_counts_year_files = save_figure(
        fig,
        FIGURE_DIR / "ostrom_counts_per_year",
        caption=ostrom_counts_year_caption,
        notes=ostrom_counts_year_notes,
        source=SOURCE_NOTE,
    )
    plt.show()

show_caption(ostrom_counts_year_caption, notes=ostrom_counts_year_notes)
for path in ostrom_counts_year_files:
    print(f"Saved: {path}")

In [ ]:
# Share of publications citing Ostrom, per year. Years without publications have no bar.
ostrom_share_years = ostrom_by_year.dropna(subset=["Share"])

with econ_style():
    fig, ax = plt.subplots(figsize=figure_size("text", ratio=0.55))

    bar(ax, ostrom_share_years["Year"], ostrom_share_years["Share"] * 100, width=0.75)
    # The benchmark goes in the legend: bars at 100% would collide with free text.
    reference_line(
        ax,
        ostrom_share * 100,
        label=f"All years: {ostrom_share:.1%} ({n_citing_ostrom}/{n_source_papers})",
        in_legend=True,
    )
    ax.set_xlabel("Year")
    ax.set_ylabel("Share of publications (%)")
    year_ticks(ax, ostrom_first_year, ostrom_last_year)
    ax.set_ylim(0, 112)
    ax.set_yticks(range(0, 101, 20))
    percent_ticks(ax, "y", xmax=100)
    ax.legend(loc="upper right")

    ostrom_share_year_caption = "Share of publications citing Elinor Ostrom, per year"
    ostrom_share_year_notes = (
        "Share of each year's publications that cite Elinor Ostrom (at least one extracted "
        "reference matched to a canonical Elinor Ostrom author cluster; each publication "
        "counts once). The dashed line marks the share across all years "
        f"({n_citing_ostrom} of {n_source_papers} publications, {ostrom_share:.1%}). Years "
        "with no publications have no bar. Annual denominators are small (at most "
        f"{int(ostrom_by_year['Publications'].max())} publications), so year-to-year "
        "movements are largely noise; the five-year version smooths them."
    )
    ostrom_share_year_files = save_figure(
        fig,
        FIGURE_DIR / "ostrom_share_per_year",
        caption=ostrom_share_year_caption,
        notes=ostrom_share_year_notes,
        source=SOURCE_NOTE,
    )
    plt.show()

show_caption(ostrom_share_year_caption, notes=ostrom_share_year_notes)
for path in ostrom_share_year_files:
    print(f"Saved: {path}")

In [ ]:
import re
import textwrap


# The dedup stage can leave citation variants of one work apart (a truncated title with
# an initial-only author). Treat two titles as one work when the normalised shorter title
# is a prefix of the longer one (at least 15 characters), and print any merge for review.
def ostrom_title_key(title):
    return re.sub(r"[^a-z0-9]+", " ", str(title).casefold()).strip()


ostrom_title_roots = []
ostrom_key_to_root = {}
for title_key in sorted(set(ostrom_context["cited_reference_title"].map(ostrom_title_key)), key=len):
    root = next(
        (r for r in ostrom_title_roots if len(r) >= 15 and title_key.startswith(r)), title_key
    )
    if root == title_key:
        ostrom_title_roots.append(title_key)
    ostrom_key_to_root[title_key] = root

ostrom_context_keyed = ostrom_context.assign(
    Title_key=ostrom_context["cited_reference_title"].map(ostrom_title_key).map(ostrom_key_to_root)
)
ostrom_variant_groups = (
    ostrom_context_keyed.groupby("Title_key")["cited_reference_id"].unique().map(list)
)
merged_ostrom_variants = ostrom_variant_groups[ostrom_variant_groups.map(len) > 1]
if merged_ostrom_variants.empty:
    print("No citation variants needed merging.")
else:
    print("Citation variants merged for this figure:")
    for work_ids in merged_ostrom_variants:
        print("  " + " + ".join(work_ids))

# Distinct core publications citing each Ostrom work (a publication counts once per work).
ostrom_work_table = (
    ostrom_context_keyed.groupby("Title_key")
    .agg(
        Citing_core_papers=("citing_paper_id", "nunique"),
        Title=("cited_reference_title", lambda titles: max(titles, key=len)),
        Year=("cited_reference_year", "max"),
    )
    .reset_index(drop=True)
    .sort_values(["Citing_core_papers", "Year", "Title"], ascending=[False, True, True])
    .reset_index(drop=True)
)
display(ostrom_work_table.head(10))

# Ascending order puts the most-cited work at the top of the horizontal bar chart.
top_ostrom_works = ostrom_work_table.head(5).iloc[::-1]


def ostrom_work_label(title, year, width=44):
    """Wrap a title to at most two lines and append the year, for a y-axis tick label."""
    lines = textwrap.wrap(str(title).strip().rstrip("."), width=width) or [""]
    if len(lines) > 2:
        lines = lines[:2]
        lines[1] = lines[1].rstrip(",;:") + "…"
    if pd.notna(year) and int(year) > 0:
        lines[-1] += f" ({int(year)})"
    return "\n".join(lines)


with econ_style():
    fig, ax = plt.subplots(figsize=figure_size("text", ratio=0.5))

    work_positions = list(range(len(top_ostrom_works)))
    work_bars = barh(
        ax,
        work_positions,
        top_ostrom_works["Citing_core_papers"],
        height=0.7,
    )
    value_labels(ax, work_bars, padding=3)
    ax.set_yticks(work_positions)
    ax.set_yticklabels(
        [
            ostrom_work_label(title, year)
            for title, year in zip(top_ostrom_works["Title"], top_ostrom_works["Year"], strict=True)
        ]
    )
    ax.tick_params(axis="y", length=0)
    ax.set_xlabel("Number of core publications citing the work")
    integer_ticks(ax, "x")
    ax.set_xlim(0, top_ostrom_works["Citing_core_papers"].max() * 1.15)
    ax.margins(y=0.05)

    ostrom_works_caption = "Elinor Ostrom works most cited by the core publications"
    ostrom_works_notes = (
        "The five works by Elinor Ostrom cited by the largest number of core publications. "
        "A core publication citing a work counts once, however many times it cites it. "
        f"In total, {n_citing_ostrom} of {n_source_papers} core publications cite at least "
        f"one of {len(ostrom_work_table):,} distinct Ostrom works. Counts use the citation "
        "graph's canonical works, with citation variants of the same title merged."
    )
    ostrom_works_files = save_figure(
        fig,
        FIGURE_DIR / "ostrom_top_cited_works",
        caption=ostrom_works_caption,
        notes=ostrom_works_notes,
        source=SOURCE_NOTE,
    )
    plt.show()

show_caption(ostrom_works_caption, notes=ostrom_works_notes)
for path in ostrom_works_files:
    print(f"Saved: {path}")

## 4. Publications by author

This subsection ranks every canonical author of the extracted source publications. “Publications” counts distinct papers, “Share of corpus” divides that count by all source publications, and “Solo-authored publications” counts papers with exactly one canonical author. Shares intentionally do not sum to 100% because coauthored papers contribute once to each author.

The figure focuses on the 20 most prolific authors; the complete table below it retains all source-publication authors, including those with only one publication.

In [ ]:
from citegraph import CitationGraph

replication_graph = CitationGraph.from_out_dir(OUT_DIR)
if not replication_graph.has_authors:
    raise RuntimeError("Run the author-normalization stage before this subsection.")

source_author_occurrences = replication_graph.author_citations.loc[
    replication_graph.author_citations["record_id"].isin(replication_graph.core.index),
    ["author_id", "record_id"],
].drop_duplicates()

if source_author_occurrences.empty:
    raise ValueError("No source-paper author occurrences were found in author_citations.csv.")

paper_team_sizes = (
    source_author_occurrences.groupby("record_id")["author_id"]
    .nunique()
    .rename("Team_size")
)
source_author_occurrences = source_author_occurrences.join(paper_team_sizes, on="record_id")
source_author_occurrences["Solo_authored"] = source_author_occurrences["Team_size"].eq(1)

n_total_publications = replication_graph.n_core_works
author_publication_table = (
    source_author_occurrences.groupby("author_id")
    .agg(
        Publications=("record_id", "nunique"),
        Solo_authored_publications=("Solo_authored", "sum"),
    )
    .join(replication_graph.authors[["display_name"]], how="left")
    .rename(columns={"display_name": "Author"})
    .reset_index()
)
author_publication_table["Share_of_corpus_pct"] = (
    author_publication_table["Publications"] / n_total_publications * 100
)
author_publication_table = author_publication_table.sort_values(
    ["Publications", "Solo_authored_publications", "Author"],
    ascending=[False, False, True],
).reset_index(drop=True)
author_publication_table["Rank"] = (
    author_publication_table["Publications"].rank(method="dense", ascending=False).astype(int)
)
author_publication_table = author_publication_table[[
    "Rank",
    "author_id",
    "Author",
    "Publications",
    "Share_of_corpus_pct",
    "Solo_authored_publications",
]]

author_table_csv = OUT_DIR / "figures" / "publications_by_author.csv"
author_table_xlsx = OUT_DIR / "figures" / "publications_by_author.xlsx"
author_table_csv.parent.mkdir(parents=True, exist_ok=True)
author_publication_table.to_csv(author_table_csv, index=False)
author_publication_table.to_excel(author_table_xlsx, index=False)

print(f"Source publications: {n_total_publications:,}")
print(f"Canonical source-publication authors: {len(author_publication_table):,}")
print(f"Authors with at least one solo-authored publication: {(author_publication_table['Solo_authored_publications'] > 0).sum():,}")
print(f"Saved CSV table: {author_table_csv}")
print(f"Saved Excel table: {author_table_xlsx}")

In [ ]:
# Ascending sort puts the most prolific author at the top of a horizontal bar chart.
plot_authors = author_publication_table.head(20).sort_values(
    ["Publications", "Solo_authored_publications", "Author"],
    ascending=[True, True, False],
)
coauthored_counts = plot_authors["Publications"] - plot_authors["Solo_authored_publications"]
author_positions = range(len(plot_authors))

with econ_style():
    fig, ax = plt.subplots(figsize=figure_size("text", ratio=0.78))

    # Two series, told apart by gray fill vs. hatched white so grayscale print works.
    barh(ax, author_positions, coauthored_counts, height=0.7, label="Coauthored")
    barh(
        ax,
        author_positions,
        plot_authors["Solo_authored_publications"],
        left=coauthored_counts,
        height=0.7,
        color="white",
        hatch=HATCHES[1],
        label="Solo-authored",
    )
    ax.set_yticks(list(author_positions))
    ax.set_yticklabels(plot_authors["Author"])
    ax.tick_params(axis="y", length=0)

    # A short ranking: the exact count and corpus share at each bar end are informative.
    for y_position, (_, row) in zip(author_positions, plot_authors.iterrows(), strict=True):
        ax.text(
            row["Publications"] + 0.3,
            y_position,
            f"{int(row['Publications'])} ({row['Share_of_corpus_pct']:.1f}%)",
            va="center",
            ha="left",
            fontsize="small",
        )

    ax.set_xlabel("Number of source publications")
    integer_ticks(ax, "x")
    ax.set_xlim(0, plot_authors["Publications"].max() * 1.2)
    ax.margins(y=0.02)
    ax.legend(loc="lower right")

    figure3_caption = "Most prolific authors of the source publications"
    figure3_notes = (
        f"The twenty canonical authors with the most source publications, out of "
        f"{len(author_publication_table):,} authors and {n_total_publications:,} source "
        "publications. Each bar counts distinct publications, split into coauthored and "
        "solo-authored. Labels give the count and its share of all source publications; "
        "shares do not sum to 100 percent because a coauthored publication counts once "
        "for each of its authors."
    )
    figure3_files = save_figure(
        fig,
        FIGURE_DIR / "top20_publications_by_author",
        caption=figure3_caption,
        notes=figure3_notes,
        source=SOURCE_NOTE,
    )
    plt.show()

show_caption(figure3_caption, notes=figure3_notes)
for path in figure3_files:
    print(f"Saved: {path}")

In [ ]:
# # Complete all-author table. The CSV saved above is convenient for filtering and manuscript work.
# author_table_display = (
#     author_publication_table.rename(columns={
#         "author_id": "Author ID",
#         "Share_of_corpus_pct": "Share of corpus (%)",
#         "Solo_authored_publications": "Solo-authored publications",
#     })
#     .style
#     .hide(axis="index")
#     .format({"Share of corpus (%)": "{:.1f}"})
#     .bar(subset=["Publications"], color="#DCE8F0")
#     .set_properties(**{
#         "font-family": "-apple-system, BlinkMacSystemFont, 'Segoe UI', sans-serif",
#         "font-size": "12px",
#         "padding": "6px 10px",
#         "border": "none",
#     })
#     .set_table_styles([
#         {
#             "selector": "caption",
#             "props": [
#                 ("caption-side", "top"),
#                 ("font-size", "16px"),
#                 ("font-weight", "700"),
#                 ("text-align", "left"),
#                 ("color", "#222222"),
#                 ("padding-bottom", "10px"),
#             ],
#         },
#         {
#             "selector": "thead th",
#             "props": [
#                 ("font-weight", "700"),
#                 ("text-align", "left"),
#                 ("border-top", "2px solid #222222"),
#                 ("border-bottom", "1px solid #222222"),
#                 ("background", "white"),
#             ],
#         },
#         {
#             "selector": "tbody tr:last-child td",
#             "props": [("border-bottom", "2px solid #222222")],
#         },
#         {
#             "selector": "tbody tr:hover",
#             "props": [("background-color", "#243A41")],
#         },
#     ])
#     .set_caption(
#         f"Publications by author — all {len(author_publication_table):,} canonical source-paper authors"
#     )
# )

# display(author_table_display)

## 5. Most-cited authors in the core corpus

This subsection ranks authors of canonical works cited by the PDF-backed core. One citation is one unique `citing work → cited work` edge. If ten core works cite the same work, each canonical author receives ten credits; if one core work cites three works by the same author, that author receives three. Repeated in-text mentions are not counted.

Citation credit uses **full counting**: every coauthor of a cited work receives one credit for that edge. Cited core works are included.

In [ ]:
from citegraph import CitationGraph

replication_graph = CitationGraph.from_out_dir(OUT_DIR)
if not replication_graph.has_authors:
    raise RuntimeError("Run the author-normalization stage before this subsection.")

# Map each canonical work to its canonical authors; roles come from graph position/ring.
cited_work_authors = (
    replication_graph.author_citations[["author_id", "record_id"]]
    .drop_duplicates()
    .rename(columns={"record_id": "cited_id"})
)
all_graph_edges = (
    replication_graph.edges.loc[
        replication_graph.edges["citing_id"].isin(replication_graph.core.index),
        ["citing_id", "cited_id"],
    ]
    .drop_duplicates()
)

# Full counting: each core-originating edge gives one credit to every author of the cited work.
author_citation_edges = (
    all_graph_edges
    .merge(cited_work_authors, on="cited_id", how="inner", validate="many_to_many")
    .drop_duplicates(["author_id", "citing_id", "cited_id"])
)

if author_citation_edges.empty:
    raise ValueError("No citation edges could be linked to canonical work authors.")

author_citation_table = (
    author_citation_edges.groupby("author_id")
    .size()
    .rename("Total_citations")
    .to_frame()
    .join(replication_graph.authors[["display_name"]], how="left")
    .rename(columns={"display_name": "Author"})
    .reset_index()
    .sort_values(["Total_citations", "Author"], ascending=[False, True])
    .reset_index(drop=True)
)
author_citation_table["Rank"] = (
    author_citation_table["Total_citations"].rank(method="dense", ascending=False).astype(int)
)
author_citation_table = author_citation_table[[
    "Rank",
    "author_id",
    "Author",
    "Total_citations",
]]

attributed_edge_count = all_graph_edges["cited_id"].isin(cited_work_authors["cited_id"]).sum()
unattributed_edge_count = len(all_graph_edges) - int(attributed_edge_count)

FIGURE_DIR = OUT_DIR / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
author_citations_csv = FIGURE_DIR / "total_citations_by_author.csv"
author_citation_table.to_csv(author_citations_csv, index=False)

print(f"Core works represented: {replication_graph.n_core_works:,}")
print(f"Unique work-to-work citation edges: {len(all_graph_edges):,}")
print(f"Canonical cited authors ranked: {len(author_citation_table):,}")
print(f"Citation edges without a canonical author: {unattributed_edge_count:,}")
print(f"Saved complete table: {author_citations_csv}")

In [ ]:
plot_citations = author_citation_table.head(20).sort_values(
    ["Total_citations", "Author"],
    ascending=[True, False],
)

with econ_style():
    fig, ax = plt.subplots(figsize=figure_size("text", ratio=0.78))

    citation_bars = barh(
        ax,
        plot_citations["Author"],
        plot_citations["Total_citations"],
        height=0.7,
    )
    value_labels(ax, citation_bars, padding=3)
    ax.tick_params(axis="y", length=0)
    ax.set_xlabel("Citations received from source publications")
    integer_ticks(ax, "x")
    ax.set_xlim(0, plot_citations["Total_citations"].max() * 1.12)
    ax.margins(y=0.02)

    figure4_caption = "Most-cited authors in the source publications' bibliographies"
    figure4_notes = (
        "The twenty canonical authors receiving the most citations from the source "
        f"publications, out of {len(author_citation_table):,} cited authors. One citation "
        "is one unique citing-work → cited-work edge; under full counting every coauthor "
        "of a cited work receives one credit per edge, and repeated in-text mentions are "
        f"not counted. {len(all_graph_edges):,} edges originate from source publications, "
        f"of which {unattributed_edge_count:,} could not be linked to a canonical author."
    )
    figure4_files = save_figure(
        fig,
        FIGURE_DIR / "top20_total_citations_by_author",
        caption=figure4_caption,
        notes=figure4_notes,
        source=SOURCE_NOTE,
    )
    plt.show()

show_caption(figure4_caption, notes=figure4_notes)
for path in figure4_files:
    print(f"Saved: {path}")

In [ ]:
# # Complete table for inspecting the long tail of cited authors.
# author_citation_display = (
#     author_citation_table.rename(columns={
#         "author_id": "Author ID",
#         "Total_citations": "Total citations",
#     })
#     .style
#     .hide(axis="index")
#     .bar(subset=["Total citations"], color="#DCE8F0")
#     .set_properties(**{
#         "font-family": "-apple-system, BlinkMacSystemFont, 'Segoe UI', sans-serif",
#         "font-size": "12px",
#         "padding": "6px 10px",
#         "border": "none",
#     })
#     .set_table_styles([
#         {
#             "selector": "caption",
#             "props": [
#                 ("caption-side", "top"),
#                 ("font-size", "16px"),
#                 ("font-weight", "700"),
#                 ("text-align", "left"),
#                 ("color", "#222222"),
#                 ("padding-bottom", "10px"),
#             ],
#         },
#         {
#             "selector": "thead th",
#             "props": [
#                 ("font-weight", "700"),
#                 ("text-align", "left"),
#                 ("border-top", "2px solid #222222"),
#                 ("border-bottom", "1px solid #222222"),
#                 ("background", "white"),
#             ],
#         },
#         {
#             "selector": "tbody tr:last-child td",
#             "props": [("border-bottom", "2px solid #222222")],
#         },
#         {
#             "selector": "tbody tr:hover",
#             "props": [("background-color", "#F3F6F7")],
#         },
#     ])
#     .set_caption(
#         f"Total citations by author — all {len(author_citation_table):,} cited canonical authors"
#     )
# )

# display(author_citation_display)

## 6. Distinct cited works per author

This figure counts unique canonical works attributed to each author and cited by at least one work in the graph. A work cited many times still counts once here, making this metric different from total citations above. Cited core works are included.

“Works” covers journal articles, books, reports, working papers, and theses; the current schema does not classify publication type.

In [ ]:
from citegraph import CitationGraph

replication_graph = CitationGraph.from_out_dir(OUT_DIR)
if not replication_graph.has_authors:
    raise RuntimeError("Run the author-normalization stage before this subsection.")

cited_work_authors = (
    replication_graph.author_citations[["author_id", "record_id"]]
    .drop_duplicates()
    .rename(columns={"record_id": "cited_id"})
)
core_edges = replication_graph.edges[
    replication_graph.edges["citing_id"].isin(replication_graph.core.index)
]
cited_work_ids = set(core_edges["cited_id"].dropna())
cited_work_authors = cited_work_authors.loc[
    cited_work_authors["cited_id"].isin(cited_work_ids)
]

if cited_work_authors.empty:
    raise ValueError("No cited works could be linked to canonical authors.")

distinct_work_counts = (
    cited_work_authors.groupby("author_id")["cited_id"]
    .nunique()
    .rename("Distinct_cited_works")
    .to_frame()
    .join(replication_graph.authors[["display_name"]], how="left")
    .rename(columns={"display_name": "Author"})
    .reset_index()
    .sort_values(["Distinct_cited_works", "Author"], ascending=[False, True])
    .reset_index(drop=True)
)
plot_distinct_works = distinct_work_counts.head(20).sort_values(
    ["Distinct_cited_works", "Author"],
    ascending=[True, False],
)

with econ_style():
    fig, ax = plt.subplots(figsize=figure_size("text", ratio=0.78))

    work_bars = barh(
        ax,
        plot_distinct_works["Author"],
        plot_distinct_works["Distinct_cited_works"],
        height=0.7,
    )
    value_labels(ax, work_bars, padding=3)
    ax.tick_params(axis="y", length=0)
    ax.set_xlabel("Distinct cited works")
    integer_ticks(ax, "x")
    ax.set_xlim(0, plot_distinct_works["Distinct_cited_works"].max() * 1.12)
    ax.margins(y=0.02)

    figure5_caption = "Authors with the most distinct works cited by the source publications"
    figure5_notes = (
        "The twenty canonical authors with the most distinct cited works, out of "
        f"{len(distinct_work_counts):,} cited authors and {len(cited_work_ids):,} canonical "
        "works cited by at least one source publication. A work cited many times counts "
        "once, so this ranking differs from total citations received. Works include "
        "journal articles, books, reports, working papers, and theses; publication type "
        "is not classified."
    )
    figure5_files = save_figure(
        fig,
        FIGURE_DIR / "top20_distinct_cited_works_by_author",
        caption=figure5_caption,
        notes=figure5_notes,
        source=SOURCE_NOTE,
    )
    plt.show()

show_caption(figure5_caption, notes=figure5_notes)
for path in figure5_files:
    print(f"Saved: {path}")

## 7. The author co-citation network

The previous subsections count citations. This one asks a different question: how are
the authors the corpus cites arranged relative to one another, and where does the focal
author sit in that arrangement? That is a question about a position in a structure, not
about a total, so the first decision is which network to measure it on.

It cannot be the raw work-level citation graph. That graph is one hop deep by
construction: the pipeline extracts bibliographies from the source PDFs, so only source
publications have outgoing citations, and every cited work is a dead end whether or not
it cites anything in the real world. Path-based measures such as betweenness or closeness
would describe that sampling design rather than the field.

The standard answer in bibliometrics is to project the citation data into an **author
co-citation network** (White and Griffith 1981; White and McCain 1998, following Small
1973 on document co-citation). Two authors are joined when the same bibliography cites
both of them, and the tie is weighted by how many bibliographies do so. Co-citation is a
statement made by the citing authors: it records which bodies of work the field treats as
belonging to the same conversation. The result is a genuine one-mode network, so the
classical centrality measures are well defined on it.

The subsection reports four things, in this order.

- **The shape of the network.** Its density, and the size of the largest set of
  most-cited authors in which every pair is co-cited. On this corpus that set is large,
  which says that the literature the corpus draws on is one integrated conversation rather
  than several separate schools: the "invisible college" of Price (1965) and Crane (1972).
- **A map** of the most central authors, drawn after two steps that are standard in
  author co-citation mapping (McCain 1990). Ties are normalised by how often each author
  is cited at all (Salton's cosine; van Eck and Waltman 2009), so that a strong tie means
  distinctive pairing rather than mere frequency, and only each author's strongest
  normalised ties are kept. A second drawing of the same network positions authors
  from all their normalised ties, in the tradition of the multidimensional-scaling maps of
  author co-citation analysis (McCain 1990), and shades the three neighbourhoods found by
  modularity maximisation, named by us.
- **One centrality measure**: Freeman's (1979) *degree centrality*, the share of the other
  authors in the network with whom an author is co-cited at least once. It is the simplest
  statement of centrality and needs no definition beyond that sentence. In a network this
  dense it saturates at the top: the most-cited authors are co-cited with nearly everyone,
  so the top of the ranking is close to the citation ranking of section 5. That is stated as
  a finding rather than hidden. As a check, *normalised strength*, the sum of an author's
  cosine-normalised ties (Barrat et al. 2004; Opsahl, Agneessens and Skvoretz 2010), is
  computed alongside; it selects the same twenty authors and is kept in the exported table.
  Eigenvector centrality (Bonacich 1972) is not reported because on these data it ranks
  almost identically to degree, and betweenness is not reported because a network in which
  the most-cited authors are all mutually tied has no structural holes to bridge.
- **A self-citation check.** An author who wrote a large share of the source publications
  could be central simply because his own bibliographies cite him. The whole construction
  is repeated on the subset of source publications he did not author, and the two rankings
  are compared.

The 180 thesis-series records described in `DATA_QUALITY_NOTES.md` are excluded from
every network in this section and the next. Only one document cites them, so in practice
they never reach the network, but excluding them makes that a property of the data rather
than an accident of the inclusion threshold.

The methodology is written up in plain words, with references, in
`paper4/NETWORK_METHODOLOGY.md`.

In [ ]:
import math

import networkx as nx

from citegraph import CitationGraph

# An author enters the co-citation network once at least this many source
# bibliographies cite them. It prunes the long tail of once-cited names, which
# would otherwise dominate the node count without informing the structure. The
# top-20 ranking is the same at 2, 3 or 5; the threshold mainly sets the network size.
CENTRALITY_MIN_PAPERS = 3

# An editable install only reaches a *new* process. A kernel that already
# imported citegraph keeps the module it loaded at import time, so a method
# added to the source since then is missing until the kernel restarts.
if not hasattr(CitationGraph, "author_cocitation_network"):
    raise RuntimeError(
        "This kernel holds a citegraph imported before author_cocitation_network() "
        "was added. Restart the kernel and run the notebook from the top."
    )

full_graph = CitationGraph.from_out_dir(OUT_DIR)
if not full_graph.has_authors:
    raise RuntimeError("Run the author-normalization stage before this subsection.")

# The thesis-series records (DATA_QUALITY_NOTES.md, problem 1) were removed from
# the corpus itself on 2026-09-13, so the exclusion filter that used to stand here
# is gone. This guard fails loudly if they ever return, rather than letting 180
# phantom works quietly rejoin the network.
n_thesis_series = int(
    full_graph.works["Journal"]
    .astype(str)
    .str.contains("Ekonomiska Studier", case=False, na=False)
    .sum()
)
if n_thesis_series:
    raise RuntimeError(
        f"{n_thesis_series} thesis-series records are back in the corpus. See "
        "DATA_QUALITY_NOTES.md problem 1 before trusting any figure below."
    )
centrality_graph = full_graph

# Resolve the focus author precisely: several initial-only clusters share the
# surname, so take the one that actually authors source publications.
focus_candidates = centrality_graph.find_author("cardenas")
if focus_candidates.empty:
    raise ValueError("No canonical author matching 'Cardenas' was found.")
focus_author_id = focus_candidates["n_core_works"].idxmax()
focus_author_name = centrality_graph.authors.loc[focus_author_id, "display_name"]
print("\nCandidate clusters (the one with source publications is used):")
print(focus_candidates[["display_name", "n_works", "n_core_works", "n_citations_received"]])

# Source publications the focus author did not write. Rebuilding the whole
# construction on this subset is the self-citation robustness check.
focus_authored_sources = set(
    centrality_graph.author_citations.loc[
        (centrality_graph.author_citations["author_id"] == focus_author_id)
        & (centrality_graph.author_citations["record_id"].isin(centrality_graph.core.index)),
        "record_id",
    ]
)
independent_sources = [
    work_id for work_id in centrality_graph.core.index if work_id not in focus_authored_sources
]


def cosine_weight(network, first, second):
    """Cosine-normalised co-citation between two authors.

    Shared bibliographies divided by the geometric mean of the two authors'
    citing-bibliography counts (Salton's cosine; van Eck and Waltman 2009). Raw
    co-citation counts largely reproduce citation frequency; this asks how often
    the two are cited *together*, given how often each is cited at all. The same
    normalisation draws the map below and sums into the centrality measure.
    """
    if not network.has_edge(first, second):
        return 0.0
    shared = network[first][second]["weight"]
    n_first = network.nodes[first]["n_citing_papers"]
    n_second = network.nodes[second]["n_citing_papers"]
    return shared / math.sqrt(n_first * n_second)


def largest_mutually_tied_top(network):
    """Largest k such that the k most-cited authors are each co-cited with every other."""
    order = sorted(network, key=lambda author: -network.nodes[author]["n_citing_papers"])
    k = 1
    while k < len(order) and all(network.has_edge(order[k], other) for other in order[:k]):
        k += 1
    return k


def centrality_table(graph, citing_ids=None):
    """Degree and normalised strength over the author co-citation network.

    Degree centrality is the reported measure. Normalised strength, the sum of
    cosine-normalised ties, is computed alongside as a check on the ranking.
    """
    network = graph.author_cocitation_network(
        min_papers=CENTRALITY_MIN_PAPERS, citing_ids=citing_ids
    )
    normalised_strength = {
        author: sum(cosine_weight(network, author, other) for other in network[author])
        for author in network
    }
    table = pd.DataFrame(
        {
            "Author": pd.Series(nx.get_node_attributes(network, "display_name")),
            "Citing_publications": pd.Series(nx.get_node_attributes(network, "n_citing_papers")),
            "Degree": pd.Series(dict(network.degree())),
            "Degree_centrality": pd.Series(nx.degree_centrality(network)),
            "Normalised_strength": pd.Series(normalised_strength),
        }
    )
    table["Degree_rank"] = table["Degree_centrality"].rank(ascending=False, method="min").astype(int)
    table["Normalised_strength_rank"] = (
        table["Normalised_strength"].rank(ascending=False, method="min").astype(int)
    )
    return network, table.sort_values("Degree_centrality", ascending=False)


cocitation_network, centrality = centrality_table(centrality_graph)
_, centrality_independent = centrality_table(centrality_graph, citing_ids=independent_sources)

n_source_publications = len(centrality_graph.core)
n_network_authors = cocitation_network.number_of_nodes()
network_density = nx.density(cocitation_network)
n_mutually_tied = largest_mutually_tied_top(cocitation_network)
def citation_rank_correlation(table, measure):
    return table[measure].corr(table["Citing_publications"], method="spearman")


most_cited_50 = centrality.nlargest(50, "Citing_publications")
degree_vs_citations = citation_rank_correlation(centrality, "Degree_centrality")
degree_vs_citations_top = citation_rank_correlation(most_cited_50, "Degree_centrality")
strength_vs_citations = citation_rank_correlation(centrality, "Normalised_strength")
strength_vs_citations_top = citation_rank_correlation(most_cited_50, "Normalised_strength")
focus_row = centrality.loc[focus_author_id]
focus_row_independent = centrality_independent.loc[focus_author_id]

print(
    f"\nCo-citation network: {n_network_authors:,} authors, "
    f"{cocitation_network.number_of_edges():,} co-citation ties, density {network_density:.2f}."
)
print(
    f"The {n_mutually_tied} most-cited authors are each co-cited with every other "
    f"(a complete clique), so degree centrality saturates at the top. Rank correlation with "
    f"citation counts: degree {degree_vs_citations:.2f} over all authors and "
    f"{degree_vs_citations_top:.2f} among the 50 most-cited; normalised strength "
    f"{strength_vs_citations:.2f} and {strength_vs_citations_top:.2f}."
)
top20_degree = set(centrality.nsmallest(20, "Degree_rank").index)
top20_strength = set(centrality.nsmallest(20, "Normalised_strength_rank").index)
n_top20_shared = len(top20_degree & top20_strength)
degree_vs_strength = centrality["Degree_centrality"].corr(
    centrality["Normalised_strength"], method="spearman"
)
if n_top20_shared == 20:
    strength_check_sentence = "selects the same 20 authors"
else:
    strength_check_sentence = f"selects {n_top20_shared} of the same 20 authors"
print(
    f"Normalised strength, computed as a check, {strength_check_sentence} for the top twenty "
    f"(rank correlation with degree {degree_vs_strength:.2f} over all authors)."
)

print(
    f"\n{focus_author_name}: degree centrality {focus_row['Degree_centrality']:.3f}, co-cited with "
    f"{focus_row['Degree']} of {n_network_authors - 1} authors (rank {focus_row['Degree_rank']}); "
    f"normalised strength {focus_row['Normalised_strength']:.1f} (rank "
    f"{focus_row['Normalised_strength_rank']}); cited by {focus_row['Citing_publications']} of "
    f"{n_source_publications} source publications."
)
print(
    f"Excluding his own {len(focus_authored_sources)} source publications "
    f"({len(independent_sources)} remain, {centrality_independent.shape[0]} authors in the "
    f"network): degree rank {focus_row_independent['Degree_rank']}, normalised strength rank "
    f"{focus_row_independent['Normalised_strength_rank']}, cited by "
    f"{focus_row_independent['Citing_publications']} of {len(independent_sources)}."
)

centrality_export = centrality.rename_axis("author_id").reset_index()
centrality_export.to_csv(OUT_DIR / "author_centrality.csv", index=False)
print(f"\nSaved: {OUT_DIR / 'author_centrality.csv'}")
centrality.head(20)

In [ ]:
# Shared drawing helpers for the two network figures below.
import math

author_surnames = centrality_graph.authors["surname"].fillna(
    centrality_graph.authors["display_name"]
)
citing_counts = centrality["Citing_publications"]


def short_label(author_id):
    """Surname alone: full names collide badly in a network drawing."""
    return str(author_surnames.get(author_id, author_id))


def _label_placements(node_size):
    """Label anchors around a node: above, below, right, left.

    The offset is the node's drawn radius plus a small gap, so a label never
    lands on top of the mark it names. ``node_size`` is a matplotlib scatter
    size in points squared, which is what networkx takes.
    """
    gap = math.sqrt(node_size / math.pi) + 3.5
    return (
        (0, gap, "center", "bottom"),
        (0, -gap, "center", "top"),
        (gap, 0, "left", "center"),
        (-gap, 0, "right", "center"),
    )


def draw_node_labels(ax, positions, node_sizes, focus_id, *, fontsize=7):
    """Label each node, moving the label around it when names would collide.

    A force-directed layout routinely places two nodes close enough that their
    names overlap. Each label is drawn, measured against the labels already on
    the figure, and moved to the next anchor around its node when it clashes.
    Measuring the rendered text rather than guessing at distances is what makes
    this hold for a different corpus, or the same corpus one stage later.
    Authors cited by more publications are placed first, so the most prominent
    names keep the preferred position above the node.
    """
    figure = ax.figure
    figure.canvas.draw()
    renderer = figure.canvas.get_renderer()

    def place(author_id, placement):
        dx, dy, ha, va = placement
        return ax.annotate(
            short_label(author_id),
            positions[author_id],
            xytext=(dx, dy),
            textcoords="offset points",
            ha=ha,
            va=va,
            fontsize=(fontsize + 0.5) if author_id == focus_id else fontsize,
        )

    placed_boxes = []
    for author_id in sorted(positions, key=lambda a: -citing_counts.get(a, 0)):
        placements = _label_placements(node_sizes[author_id])
        box = None
        for placement in placements:
            label = place(author_id, placement)
            # A little padding so accepted labels do not end up touching.
            box = label.get_window_extent(renderer=renderer).expanded(1.06, 1.25)
            if not any(box.overlaps(other) for other in placed_boxes):
                break
            label.remove()
            box = None
        if box is None:
            # Every anchor clashes; keep the preferred one and accept it.
            label = place(author_id, placements[0])
            box = label.get_window_extent(renderer=renderer).expanded(1.06, 1.25)
        placed_boxes.append(box)

In [ ]:
# The co-citation network is too dense to draw whole: with almost every pair of
# authors tied at least once, a force-directed layout has nothing to separate and
# collapses into a block. Two standard steps make a readable map of it. First,
# normalise each tie by how often the two authors are cited at all (Salton's
# cosine, via cosine_weight above), because raw co-citation counts largely
# reproduce citation frequency. Second, keep only each author's strongest
# normalised ties, the "backbone" of the network, which leaves a structure a
# layout can resolve.
import itertools

MAP_AUTHORS = 28
MAP_TIES_PER_AUTHOR = 3

# Nodes are the most-cited authors, the selection rule of classical author
# co-citation mapping (White and Griffith 1981). Selecting by normalised strength
# instead pulls in rarely cited authors whose few citations happen to coincide,
# which crowds the map with a tail that says little about the field.
map_authors = list(
    centrality.nlargest(MAP_AUTHORS, ["Citing_publications", "Normalised_strength"]).index
)

map_network = nx.Graph()
map_network.add_nodes_from(map_authors)
for author_id in map_authors:
    neighbours = sorted(
        (
            (other, cosine_weight(cocitation_network, author_id, other))
            for other in map_authors
            if other != author_id
        ),
        key=lambda pair: pair[1],
        reverse=True,
    )
    for other, similarity in neighbours[:MAP_TIES_PER_AUTHOR]:
        if similarity > 0:
            map_network.add_edge(author_id, other, similarity=similarity)

# How much community structure is there, honestly? Modularity is measured on the
# *unpruned* normalised ties among the mapped authors. Pruning to a backbone
# manufactures apparent structure, so a modularity computed on the drawn edges
# would overstate it. The partition is printed so the reading offered in the
# caption can be checked against it; it is not itself a reported result.
unpruned_map_ties = nx.Graph()
unpruned_map_ties.add_nodes_from(map_authors)
for first, second in itertools.combinations(map_authors, 2):
    similarity = cosine_weight(cocitation_network, first, second)
    if similarity > 0:
        unpruned_map_ties.add_edge(first, second, weight=similarity)
map_partition = nx.community.louvain_communities(unpruned_map_ties, weight="weight", seed=12)
map_modularity = nx.community.modularity(unpruned_map_ties, map_partition, weight="weight")
print(f"Modularity of the best partition of the unpruned normalised ties: {map_modularity:.2f}")
for group in sorted(map_partition, key=len, reverse=True):
    print("  ", ", ".join(short_label(a) for a in sorted(group, key=lambda a: -citing_counts[a])))

# A fixed seed keeps the drawing identical from run to run.
map_positions = nx.spring_layout(
    map_network, weight="similarity", seed=12, k=1.15, iterations=600
)

with econ_style():
    fig, ax = plt.subplots(figsize=figure_size("text", ratio=0.80))

    max_citing = citing_counts[map_authors].max()
    map_node_sizes = {a: 60 + 500 * (citing_counts[a] / max_citing) for a in map_network}
    similarities = [d["similarity"] for _, _, d in map_network.edges(data=True)]
    max_similarity = max(similarities)

    nx.draw_networkx_edges(
        map_network,
        map_positions,
        ax=ax,
        width=[0.3 + 1.7 * (s / max_similarity) for s in similarities],
        edge_color=EDGE_COLOR,
        alpha=0.75,
    )
    nx.draw_networkx_nodes(
        map_network,
        map_positions,
        ax=ax,
        node_size=[map_node_sizes[a] for a in map_network],
        node_color=["black" if a == focus_author_id else NODE_FILL for a in map_network],
        edgecolors="black",
        linewidths=0.6,
    )
    draw_node_labels(ax, map_positions, map_node_sizes, focus_author_id, fontsize=7)
    network_axes(ax, equal=False)
    ax.margins(0.12)

    map_caption = "The intellectual structure of the corpus: author co-citation network"
    map_notes = (
        f"Nodes are the {MAP_AUTHORS} authors cited by the largest number of the "
        f"{n_source_publications} source publications, placed in the author co-citation network built from those "
        f"publications' bibliographies. Two authors are tied when a source bibliography cites both. Ties "
        f"are normalised by the square root of the product of the two authors' "
        f"citing-publication counts, so that a tie reflects distinctive co-citation rather than "
        f"citation frequency, and each author keeps only their {MAP_TIES_PER_AUTHOR} strongest "
        f"normalised ties. Drawing the unpruned network would be uninformative because "
        f"{network_density:.0%} of all author pairs are tied at least once. Three groupings can "
        f"be read off the drawing: the commons-governance school around Ostrom, Walker and "
        f"Gardner; the social-preferences and cultural-evolution literature around Fehr, Bowles "
        f"and Henrich; and the field-experimental and public-goods literature around List, "
        f"Harrison and Gächter. They are our reading of the map, not the output of a clustering "
        f"algorithm: a modularity partition of the unpruned normalised ties among these authors "
        f"separates them only faintly (modularity {map_modularity:.2f}). Node area is "
        f"proportional to the number of source publications citing the author, and "
        f"{focus_author_name} is the filled node. Positions come from a force-directed layout "
        f"with a fixed seed and carry no units."
    )
    map_files = save_figure(
        fig,
        FIGURE_DIR / "cocitation_network_map",
        caption=map_caption,
        notes=map_notes,
        source=SOURCE_NOTE,
    )
    plt.show()

show_caption(map_caption, notes=map_notes)
for path in map_files:
    print(f"Saved: {path}")

In [ ]:
# A second drawing of the same map, built to be read. Three things change from the
# drawing above. Positions come from a stress (Kamada-Kawai) layout over the
# normalised ties among *all* pairs of mapped authors, in the tradition of the
# multidimensional-scaling maps of author co-citation analysis, so two authors sit
# close when the corpus cites them together relative to how often it cites each,
# whether or not a line is drawn between them. The modularity neighbourhoods
# computed above are shaded and named, and node fill follows them so the grouping
# survives grayscale print. Labels are placed by a collision search against other
# labels, nodes, drawn ties and the neighbourhood names, with a hairline leader
# when a label has to move away from its node.
import numpy as np
from matplotlib.patches import Polygon

# The neighbourhoods are a *reading* of the field, stated here by hand, not an
# output of the algorithm. Modularity maximisation over the normalised ties among
# these 28 authors separates them only faintly (see map_modularity above) and it
# is not stable: on this corpus it returns two communities, merging commons
# governance with the field-experimental group. A partition that weak should not
# decide what a figure claims, and re-deriving it every run would let the reading
# drift with each re-extraction.
#
# The grouping below is not arbitrary. It reproduces a modularity partition of the
# *raw* co-citation counts among the same authors, and it is what the three
# literatures look like to a reader of the corpus. Written as a literal, it keeps
# the map stable and the claim inspectable.
NEIGHBOURHOODS = {
    "Commons governance": [
        "Cardenas", "Gardner", "Hardin", "Janssen", "Ledyard", "Lopez",
        "Murphy", "Ostrom", "Stranlund", "Velez", "Walker", "Willis",
    ],
    "Social preferences": [
        "Andreoni", "Bowles", "Camerer", "Falk", "Fehr", "Fischbacher",
        "Gintis", "Gneezy", "G\u00e4chter", "Henrich", "Kahneman", "McCabe",
    ],
    "Experimental methods\nand public goods": [
        "Carpenter", "Croson", "Harrison", "List",
    ],
}
NEIGHBOURHOOD_FILL = {
    "Commons governance": GRAYS[3],
    "Social preferences": "white",
    "Experimental methods\nand public goods": GRAYS[1],
}
HULL_FILL, HULL_EDGE, HULL_PAD_PT = "#F2F2F2", "#C8C8C8", 18
LABEL_DIRECTIONS = [(0, 1), (0, -1), (1, 0), (-1, 0), (1, 1), (-1, 1), (1, -1), (-1, -1)]

map_authors_surnames = {a: centrality_graph.authors.loc[a, "surname"] for a in map_authors}
if len(set(map_authors_surnames.values())) != len(map_authors):
    raise RuntimeError("Two mapped authors share a surname; key NEIGHBOURHOODS on ids instead.")

# Resolve the reading onto the mapped authors by surname, and fail loudly rather
# than draw a partial map: every mapped author must be named exactly once, and
# every name must correspond to an author actually on the map.
_surname_to_id = {map_authors_surnames[a]: a for a in map_authors}
neighbourhood_of = {}
for name, surnames in NEIGHBOURHOODS.items():
    for surname in surnames:
        if surname not in _surname_to_id:
            raise RuntimeError(
                f"NEIGHBOURHOODS names {surname!r}, who is not among the {MAP_AUTHORS} "
                f"mapped authors: {sorted(map_authors_surnames.values())}"
            )
        author_id = _surname_to_id[surname]
        if author_id in neighbourhood_of:
            raise RuntimeError(f"{surname!r} appears in more than one neighbourhood.")
        neighbourhood_of[author_id] = name
_unassigned = sorted(map_authors_surnames[a] for a in map_authors if a not in neighbourhood_of)
if _unassigned:
    raise RuntimeError(
        f"These mapped authors are in no neighbourhood: {_unassigned}. "
        "Add them to NEIGHBOURHOODS."
    )


def push_apart(positions, node_sizes, fig_width_in, *, pad_pt=6.0, iterations=200):
    """Move nodes apart until no two drawn circles overlap (approximate, in layout units)."""
    coords = np.array(list(positions.values()))
    span = max(np.ptp(coords[:, 0]), np.ptp(coords[:, 1]))
    pt_per_unit = fig_width_in * 72 * 0.80 / span
    radius = {a: (math.sqrt(node_sizes[a] / math.pi) + pad_pt) / pt_per_unit for a in positions}
    for _ in range(iterations):
        moved = False
        for a, b in itertools.combinations(list(positions), 2):
            gap = positions[b] - positions[a]
            distance = float(np.linalg.norm(gap))
            needed = radius[a] + radius[b]
            if distance < needed:
                moved = True
                direction = gap / distance if distance > 1e-9 else np.array([1.0, 0.0])
                shift = (needed - distance) / 2 * direction
                positions[a] = positions[a] - shift
                positions[b] = positions[b] + shift
        if not moved:
            break
    return positions


def convex_hull(points):
    """Andrew's monotone chain; returns the hull vertices counter-clockwise."""
    pts = sorted(map(tuple, points))
    if len(pts) <= 2:
        return pts

    def cross(o, a, b):
        return (a[0] - o[0]) * (b[1] - o[1]) - (a[1] - o[1]) * (b[0] - o[0])

    lower, upper = [], []
    for pt in pts:
        while len(lower) >= 2 and cross(lower[-2], lower[-1], pt) <= 0:
            lower.pop()
        lower.append(pt)
    for pt in reversed(pts):
        while len(upper) >= 2 and cross(upper[-2], upper[-1], pt) <= 0:
            upper.pop()
        upper.append(pt)
    return lower[:-1] + upper[:-1]


def segment_hits_box(p, q, box):
    """True when the segment p-q touches the display-space bounding box."""
    x0, y0, x1, y1 = box.x0, box.y0, box.x1, box.y1
    if (x0 <= p[0] <= x1 and y0 <= p[1] <= y1) or (x0 <= q[0] <= x1 and y0 <= q[1] <= y1):
        return True

    def ccw(a, b, c):
        return (c[1] - a[1]) * (b[0] - a[0]) > (b[1] - a[1]) * (c[0] - a[0])

    def crosses(a, b, c, d):
        return ccw(a, c, d) != ccw(b, c, d) and ccw(a, b, c) != ccw(a, b, d)

    corners = [(x0, y0), (x1, y0), (x1, y1), (x0, y1)]
    return any(crosses(p, q, corners[i], corners[(i + 1) % 4]) for i in range(4))


def circle_hits_box(center, radius, box):
    nearest_x = min(max(center[0], box.x0), box.x1)
    nearest_y = min(max(center[1], box.y0), box.y1)
    return math.hypot(center[0] - nearest_x, center[1] - nearest_y) < radius


def place_labels_clear(ax, network, positions, node_sizes, focus_id, *, fontsize=7.5, reserved=()):
    """Label every node where the name collides least with what is already drawn.

    Each label tries eight anchors around its node at a near and a far offset.
    A candidate is scored by the labels it overlaps (heaviest), the other nodes
    it covers, and the drawn ties it crosses, plus a small penalty for the far
    offset and for less preferred directions. The far offset draws a hairline
    leader. Authors cited by more publications are placed first so the most
    prominent names take the clearest positions.
    """
    figure = ax.figure
    figure.canvas.draw()
    renderer = figure.canvas.get_renderer()
    px_per_pt = figure.dpi / 72
    centers = {a: ax.transData.transform(positions[a]) for a in positions}
    radii_px = {a: math.sqrt(node_sizes[a] / math.pi) * px_per_pt for a in positions}
    segments = [(u, v, centers[u], centers[v]) for u, v in network.edges()]
    placed_boxes = list(reserved)

    def draw(author, dx, dy, offset_pt, leader):
        norm = math.hypot(dx, dy)
        kwargs = {}
        if leader:
            kwargs["arrowprops"] = dict(
                arrowstyle="-", lw=0.4, color=EDGE_COLOR, shrinkA=0,
                shrinkB=math.sqrt(node_sizes[author] / math.pi),
            )
        return ax.annotate(
            short_label(author),
            positions[author],
            xytext=(dx / norm * offset_pt, dy / norm * offset_pt),
            textcoords="offset points",
            ha="center" if dx == 0 else ("left" if dx > 0 else "right"),
            va="center" if dy == 0 else ("bottom" if dy > 0 else "top"),
            fontsize=fontsize + (0.5 if author == focus_id else 0),
            zorder=5,
            **kwargs,
        )

    for author in sorted(positions, key=lambda a: -citing_counts.get(a, 0)):
        radius_pt = math.sqrt(node_sizes[author] / math.pi)
        best = None
        for far in (False, True):
            offset_pt = radius_pt + (14.0 if far else 2.5)
            for rank, (dx, dy) in enumerate(LABEL_DIRECTIONS):
                label = draw(author, dx, dy, offset_pt, leader=far)
                box = label.get_window_extent(renderer=renderer).expanded(1.08, 1.3)
                score = 10 * sum(box.overlaps(other) for other in placed_boxes)
                score += 5 * sum(
                    circle_hits_box(centers[b], radii_px[b], box) for b in positions if b != author
                )
                score += sum(segment_hits_box(p, q, box) for u, v, p, q in segments if author not in (u, v))
                score += (3.0 if far else 0.0) + 0.1 * rank
                if best is None or score < best[0]:
                    if best is not None:
                        best[1].remove()
                    best = (score, label, box)
                else:
                    label.remove()
            if best[0] < 3.0:
                break
        placed_boxes.append(best[2])


# Positions: stress layout on the normalised ties among all pairs, then push
# the drawn circles apart so no two overlap.
stress_distances = {
    a: {b: (0.0 if a == b else 1.0 - cosine_weight(cocitation_network, a, b)) for b in map_authors}
    for a in map_authors
}
neighbourhood_positions = {
    a: np.asarray(p, dtype=float)
    for a, p in nx.kamada_kawai_layout(unpruned_map_ties, dist=stress_distances, weight=None).items()
}
neighbourhood_figsize = figure_size("text", ratio=0.80)
neighbourhood_positions = push_apart(neighbourhood_positions, map_node_sizes, neighbourhood_figsize[0])

# Who does the focus author reach for, and who reaches for him from the other neighbourhoods?
focus_strongest = [
    short_label(b) for b, _ in sorted(
        ((b, cosine_weight(cocitation_network, focus_author_id, b)) for b in map_authors if b != focus_author_id),
        key=lambda pair: -pair[1],
    )[:MAP_TIES_PER_AUTHOR]
]
def strongest_partners(author_id):
    return [
        b for b, _ in sorted(
            ((b, cosine_weight(cocitation_network, author_id, b)) for b in map_authors if b != author_id),
            key=lambda pair: -pair[1],
        )[:MAP_TIES_PER_AUTHOR]
    ]


n_outside_pick_focus = sum(
    1 for a in map_authors
    if neighbourhood_of[a] != neighbourhood_of[focus_author_id]
    and focus_author_id in strongest_partners(a)
)

with econ_style():
    fig, ax = plt.subplots(figsize=neighbourhood_figsize)
    network_axes(ax, equal=False)

    layout_centre = np.mean(np.array(list(neighbourhood_positions.values())), axis=0)
    neighbourhood_name_texts = []
    for name in NEIGHBOURHOODS:
        members = [a for a in map_authors if neighbourhood_of[a] == name]
        member_points = np.array([neighbourhood_positions[a] for a in members])
        hull = convex_hull(member_points)
        # A thick round-joined stroke in the fill colour pads the hull; a slightly
        # thicker stroke underneath in the edge colour gives it a hairline outline.
        ax.add_patch(Polygon(hull, closed=True, facecolor=HULL_EDGE, edgecolor=HULL_EDGE,
                             linewidth=HULL_PAD_PT + 1.2, joinstyle="round", zorder=0))
        ax.add_patch(Polygon(hull, closed=True, facecolor=HULL_FILL, edgecolor=HULL_FILL,
                             linewidth=HULL_PAD_PT, joinstyle="round", zorder=0.1))
        # The name sits outside the hull, on the side facing away from the layout centre.
        outward = member_points.mean(axis=0) - layout_centre
        outward = outward / (np.linalg.norm(outward) or 1.0)
        extreme = max(hull, key=lambda pt: float(np.dot(np.array(pt) - layout_centre, outward)))
        neighbourhood_name_texts.append(
            ax.annotate(
                name, extreme,
                xytext=(outward[0] * (HULL_PAD_PT / 2 + 8), outward[1] * (HULL_PAD_PT / 2 + 8)),
                textcoords="offset points",
                ha="left" if outward[0] > 0.3 else ("right" if outward[0] < -0.3 else "center"),
                va="bottom" if outward[1] > 0.3 else ("top" if outward[1] < -0.3 else "center"),
                fontsize=9, color=GRAYS[0], style="italic", zorder=4,
            )
        )

    nx.draw_networkx_edges(
        map_network, neighbourhood_positions, ax=ax,
        width=[0.3 + 1.7 * (s / max_similarity) for s in similarities],
        edge_color=EDGE_COLOR, alpha=0.8,
    )
    nx.draw_networkx_nodes(
        map_network, neighbourhood_positions, ax=ax,
        node_size=[map_node_sizes[a] for a in map_network],
        node_color=[
            "black" if a == focus_author_id else NEIGHBOURHOOD_FILL[neighbourhood_of[a]]
            for a in map_network
        ],
        edgecolors="black", linewidths=0.6,
    )
    ax.margins(0.16)
    fig.canvas.draw()
    reserved_boxes = [
        text.get_window_extent(renderer=fig.canvas.get_renderer()).expanded(1.05, 1.2)
        for text in neighbourhood_name_texts
    ]
    place_labels_clear(
        ax, map_network, neighbourhood_positions, map_node_sizes, focus_author_id, reserved=reserved_boxes
    )

    neighbourhood_caption = (
        "The intellectual structure of the corpus: three neighbourhoods in the author co-citation network"
    )
    neighbourhood_notes = (
        f"Same {MAP_AUTHORS} authors and same drawn ties as the preceding map: nodes are the authors "
        f"cited by the largest number of the {n_source_publications} source publications, a line joins an author "
        f"to their {MAP_TIES_PER_AUTHOR} strongest cosine-normalised co-citation partners, line width is "
        f"proportional to the normalised tie, and node area to the number of citing source "
        f"publications. Positions differ: they come from a stress layout over the normalised ties among "
        f"all pairs, so two authors sit close when the corpus cites them together relative to how often "
        f"it cites each, whether or not a line is drawn between them. Shaded regions group the "
        f"authors into three research neighbourhoods; node fill follows the grouping. The grouping "
        f"and its names are our reading of the corpus, not an output of an algorithm: modularity "
        f"maximisation over these same normalised ties separates the authors only faintly "
        f"(modularity {map_modularity:.2f}) and merges the first and third neighbourhoods, so it is "
        f"reported here rather than used to draw the figure. The separation is genuinely faint: every "
        f"pair of authors shown is co-cited by at least one source publication. The neighbourhoods "
        f"group authors by research literature rather than by the drawn lines, so an author can sit "
        f"in one neighbourhood while their strongest drawn "
        f"ties reach "
        f"another. {focus_author_name} is the black node; his own strongest ties run to "
        f"{', '.join(focus_strongest[:-1])} and {focus_strongest[-1]}, and {n_outside_pick_focus} "
        f"authors from the other two neighbourhoods count him among their strongest partners. "
        f"Positions carry no units."
    )
    neighbourhood_files = save_figure(
        fig,
        FIGURE_DIR / "cocitation_network_map_neighbourhoods",
        caption=neighbourhood_caption,
        notes=neighbourhood_notes,
        source=SOURCE_NOTE,
    )
    plt.show()

show_caption(neighbourhood_caption, notes=neighbourhood_notes)
for path in neighbourhood_files:
    print(f"Saved: {path}")

In [ ]:
plot_centrality = centrality.head(20).sort_values("Degree_centrality")

with econ_style():
    fig, ax = plt.subplots(figsize=figure_size("text", ratio=0.78))

    bar_colors = ["black" if a == focus_author_id else GRAYS[1] for a in plot_centrality.index]
    centrality_bars = barh(
        ax,
        plot_centrality["Author"],
        plot_centrality["Degree_centrality"] * 100,
        height=0.7,
        color=bar_colors,
    )
    value_labels(ax, centrality_bars, fmt="{:,.1f}%", padding=3)
    ax.tick_params(axis="y", length=0)
    ax.set_xlabel("Degree centrality (% of authors co-cited with)")
    percent_ticks(ax, "x", xmax=100)
    ax.set_xlim(0, 108)
    ax.margins(y=0.02)

    degree_caption = "Author centrality in the co-citation network"
    degree_notes = (
        f"Freeman degree centrality in the author co-citation network, for the 20 most central "
        f"authors. The value is the share of the {n_network_authors - 1} other authors in the "
        f"network with whom the author is co-cited by at least one of the {n_source_publications} "
        f"source publications. The measure saturates at the top: the {n_mutually_tied} most-cited "
        f"authors are each co-cited with every other, so the order of the leading bars is close to "
        f"the citation ranking. Normalising each tie by the two authors' citation frequencies and "
        f"summing (normalised strength, reported in author_centrality.csv) "
        f"{strength_check_sentence}. {focus_author_name} is the filled bar. The network includes "
        f"authors cited by at least {CENTRALITY_MIN_PAPERS} source publications."
    )
    degree_files = save_figure(
        fig,
        FIGURE_DIR / "top20_degree_centrality_by_author",
        caption=degree_caption,
        notes=degree_notes,
        source=SOURCE_NOTE,
    )
    plt.show()

show_caption(degree_caption, notes=degree_notes)
for path in degree_files:
    print(f"Saved: {path}")

In [ ]:
ROBUSTNESS_AUTHORS = 15
robustness = centrality.head(ROBUSTNESS_AUTHORS)[["Author", "Degree_rank"]].copy()
robustness["Rank_independent"] = centrality_independent["Degree_rank"]
# An author central only through the focus author's own bibliographies would leave
# the restricted network entirely; that would itself be the finding.
robustness["Rank_independent"] = robustness["Rank_independent"].fillna(
    len(centrality_independent) + 1
)
robustness = robustness.sort_values("Degree_rank", ascending=False)

focus_rank_all = int(focus_row["Degree_rank"])
focus_rank_independent = int(focus_row_independent["Degree_rank"])
if focus_rank_all == focus_rank_independent:
    focus_rank_sentence = f"{focus_author_name} keeps rank {focus_rank_all} in both constructions."
else:
    focus_rank_sentence = (
        f"{focus_author_name} moves from rank {focus_rank_all} to rank {focus_rank_independent}."
    )
strength_rank_all = int(focus_row["Normalised_strength_rank"])
strength_rank_independent = int(focus_row_independent["Normalised_strength_rank"])
strength_check = (
    f"Under the normalised-strength check he ranks {strength_rank_all} with all publications "
    f"and {strength_rank_independent} without his own"
)
if strength_rank_independent != 1:
    strength_leader = centrality_independent.loc[
        centrality_independent["Normalised_strength_rank"] == 1, "Author"
    ].iloc[0]
    strength_check += f", behind {strength_leader}"
strength_check += "."

with econ_style():
    fig, ax = plt.subplots(figsize=figure_size("text", ratio=0.70))

    positions = range(len(robustness))
    ax.hlines(
        positions,
        robustness["Degree_rank"],
        robustness["Rank_independent"],
        color=GRAYS[2],
        linewidth=1.0,
        zorder=1,
    )
    # The open marker is drawn larger so an unchanged rank reads as a ring around
    # a dot rather than hiding one marker behind the other.
    ax.scatter(
        robustness["Degree_rank"],
        positions,
        s=62,
        facecolors="white",
        edgecolors="black",
        linewidths=0.9,
        zorder=2,
        label="All source publications",
    )
    ax.scatter(
        robustness["Rank_independent"],
        positions,
        s=22,
        facecolors="black",
        edgecolors="black",
        linewidths=0.9,
        zorder=3,
        label=f"Excluding publications authored by {focus_author_name.split()[-1]}",
    )
    ax.set_yticks(list(positions))
    ax.set_yticklabels(robustness["Author"])
    ax.tick_params(axis="y", length=0)
    ax.set_xlabel("Rank by degree centrality (1 = most central)")
    integer_ticks(ax, "x")
    highest_rank = robustness[["Degree_rank", "Rank_independent"]].to_numpy().max()
    ax.set_xlim(0, highest_rank + 1)
    # The top rows sit at the left, so the upper right is the only clear space.
    ax.legend(frameon=False, loc="upper right", fontsize="small")
    ax.margins(y=0.06)

    robustness_caption = "Centrality rankings are not an artefact of self-citation"
    robustness_notes = (
        f"Rank by degree centrality in the author co-citation network for the "
        f"{ROBUSTNESS_AUTHORS} most central authors, under two constructions. Open markers use "
        f"all {n_source_publications} source publications. Filled markers rebuild the network "
        f"from the {len(independent_sources)} source publications {focus_author_name} did not "
        f"author, so that no tie can arise from his own bibliographies; he remains cited by "
        f"{int(focus_row_independent['Citing_publications'])} of them. {focus_rank_sentence} "
        f"{strength_check} A marker drawn inside a ring means the rank is unchanged; a long "
        f"segment means the rank depends on which publications are included. The check removes "
        f"mechanical self-citation; it cannot remove the fact that many of the remaining "
        f"publications are by his students and co-authors."
    )
    robustness_files = save_figure(
        fig,
        FIGURE_DIR / "centrality_self_citation_robustness",
        caption=robustness_caption,
        notes=robustness_notes,
        source=SOURCE_NOTE,
    )
    plt.show()

show_caption(robustness_caption, notes=robustness_notes)
for path in robustness_files:
    print(f"Saved: {path}")

## 8. The corpus's centre of gravity over time

The network above is a snapshot: it pools 26 years of bibliographies. This subsection asks
how the intellectual base moved. It is deliberately not a network measure. With 92 source
publications spread over four periods, per-period networks would be too small to compare
structurally, so the figure is descriptive: for six authors who mark distinct positions in
the literature, the share of each period's source publications whose bibliography cites at
least one of their works.

The periods are cut so that each holds a comparable number of source publications; the
denominators are printed with the figure and repeated in its notes. The six authors are
Garrett Hardin (the tragedy-of-the-commons position the literature defined itself
against), Elinor Ostrom (the commons-governance school), the focal author, Ernst Fehr
(social preferences), John A. List (field-experimental method) and Christopher Blattman
(the economics of conflict). Identities are resolved through canonical author clusters, as
in section 3, so abbreviated citations are counted. The thesis-series records are excluded
as in section 7.

In [ ]:
from citegraph.plotting import panel_title

# Period boundaries, chosen so each period holds a comparable number of source
# publications. Labels are two-line so the ticks stay horizontal.
PERIODS = [(1998, 2007), (2008, 2013), (2014, 2019), (2020, 2024)]
# Surnames to track, in panel order. Each resolves to the cluster with the most
# distinct citing works, so an initial-only split cluster never wins.
GRAVITY_AUTHORS = ["hardin", "ostrom", "cardenas", "fehr", "list", "blattman"]
# A cluster's display name is the most frequent extracted spelling; override it
# for the panel title where the corpus spells a name unconventionally.
GRAVITY_LABELS = {"hardin": "Garrett Hardin"}


def resolve_author(graph, surname):
    """Canonical author id for a surname: exact surname match, most-cited cluster."""
    candidates = graph.find_author(surname)
    if candidates.empty:
        raise ValueError(f"No canonical author matching {surname!r} was found.")
    surname_column = "surname_norm" if "surname_norm" in candidates.columns else "surname"
    exact = candidates[candidates[surname_column].astype(str).str.casefold() == surname.casefold()]
    pool = exact if not exact.empty else candidates
    return pool["n_distinct_citing_works"].idxmax()


def period_label(year):
    for start, end in PERIODS:
        if start <= year <= end:
            return f"{start}–\n{end}"
    return None


gravity_sources = centrality_graph.core.reset_index()
gravity_sources["Year_numeric"] = pd.to_numeric(gravity_sources["Year"], errors="coerce")
gravity_sources["Period"] = gravity_sources["Year_numeric"].map(
    lambda y: period_label(int(y)) if pd.notna(y) else None
)
gravity_excluded = gravity_sources[gravity_sources["Period"].isna()]
gravity_sources = gravity_sources.dropna(subset=["Period"])
period_order = [f"{start}–\n{end}" for start, end in PERIODS]
period_sizes = gravity_sources.groupby("Period")["id"].nunique().reindex(period_order, fill_value=0)

gravity_ids = {surname: resolve_author(centrality_graph, surname) for surname in GRAVITY_AUTHORS}
gravity_shares = {}
for surname, author_id in gravity_ids.items():
    author_works = set(
        centrality_graph.author_citations.loc[
            centrality_graph.author_citations["author_id"] == author_id, "record_id"
        ]
    )
    citing_ids = set(
        centrality_graph.edges.loc[
            centrality_graph.edges["cited_id"].isin(author_works), "citing_id"
        ]
    )
    cites = gravity_sources["id"].isin(citing_ids)
    panel_name = GRAVITY_LABELS.get(surname, centrality_graph.authors.loc[author_id, "display_name"])
    gravity_shares[panel_name] = (
        cites.groupby(gravity_sources["Period"]).mean().reindex(period_order, fill_value=0.0)
    )
gravity_table = pd.DataFrame(gravity_shares)
gravity_table.insert(0, "Publications", period_sizes)
gravity_table.index = [label.replace("\n", "") for label in gravity_table.index]
gravity_table.index.name = "Period"
gravity_table.to_csv(FIGURE_DIR / "centre_of_gravity_by_period.csv")

print("Resolved clusters:", {s: a for s, a in gravity_ids.items()})
if not gravity_excluded.empty:
    print(f"{len(gravity_excluded)} source publication(s) outside the periods or without a usable year:")
    display(gravity_excluded[["id", "Title", "Year"]])
gravity_display = gravity_table.copy()
for column in gravity_display.columns.drop("Publications"):
    gravity_display[column] = gravity_display[column].map("{:.0%}".format)
display(gravity_display)

with econ_style():
    fig, axes = plt.subplots(2, 3, figsize=figure_size("text", ratio=0.62), sharey=True)

    for ax, letter, (author_name, shares) in zip(
        axes.flat, "ABCDEF", gravity_table.drop(columns="Publications").items()
    ):
        bar(ax, period_order, shares.to_numpy(), width=0.7)
        panel_title(ax, letter, author_name)
        ax.set_ylim(0, 1.05)
        percent_ticks(ax, "y", xmax=1)
        ax.tick_params(axis="x", length=0)
    for ax in axes[:, 0]:
        ax.set_ylabel("Share of the period's\npublications")

    denominators = "; ".join(
        f"{period} n={int(n)}" for period, n in period_sizes.rename(index=lambda s: s.replace("\n", "")).items()
    )
    excluded_sentence = (
        f" {len(gravity_excluded)} source publication(s) without a usable year are excluded."
        if not gravity_excluded.empty
        else ""
    )
    gravity_caption = "The corpus's centre of gravity moves twice"
    gravity_notes = (
        f"Each panel gives the share of the source publications of the period whose "
        f"bibliography cites at least one work by the author. Periods are cut to hold "
        f"comparable numbers of source publications, so the bars have different denominators: "
        f"{denominators}.{excluded_sentence} Hardin is the position the literature defined "
        f"itself against, and it stops being cited once that argument is settled. Ostrom's "
        f"commons-governance school gives way to social preferences and experimental method, "
        f"and by the last period the arrivals are from the economics of conflict. Denominators "
        f"are small, so read the direction of each panel rather than individual bars. Excludes "
        f"the 180 thesis-series records described in DATA_QUALITY_NOTES.md."
    )
    gravity_files = save_figure(
        fig,
        FIGURE_DIR / "centre_of_gravity_by_period",
        caption=gravity_caption,
        notes=gravity_notes,
        source=SOURCE_NOTE,
    )
    plt.show()

show_caption(gravity_caption, notes=gravity_notes)
for path in gravity_files:
    print(f"Saved: {path}")

## 9. The corpus's shared canon

Which works does the corpus as a whole lean on? This is the corpus-wide version of the
Ostrom ranking in section 3: instead of one author's works, it ranks every cited work by
how many of the core publications cite it, so it shows the literature the corpus holds in
common rather than the literature any one paper happens to cite.

It sits at the end rather than beside section 3 only to keep the section numbers stable --
`NETWORK_METHODOLOGY.md` refers to sections 7 and 8 of this notebook by number.


In [ ]:
import textwrap

import pandas as pd
from matplotlib.patches import Patch

from citegraph import CitationGraph

TOP_WORKS = 15

canon_graph = CitationGraph.from_out_dir(OUT_DIR)

# `top_cited` counts *distinct citing works*, and the pipeline already
# deduplicated the edge list, so a publication votes once per work it cites
# however many times its bibliography lists it. Unlike the Ostrom ranking in
# section 3 this needs no title-variant merging: checked across the top 60,
# no two canonical works prefix-merge into one.
canon_works = canon_graph.top_cited(TOP_WORKS)

# The first author's *canonical* surname rather than the spelling this record
# happened to carry -- the same person shows up as "Cardenas" and "Cárdenas"
# across records, and the author stage already resolved those into one cluster.
first_author = (
    canon_graph.author_citations[canon_graph.author_citations["position"] == 0]
    .drop_duplicates("record_id")
    .set_index("record_id")["author_id"]
)
canonical_surname = canon_graph.authors["surname"].fillna(canon_graph.authors["display_name"])


def work_label(work_id, title, year, width=42):
    """"Short title (Surname Year)" on at most two lines, for a y-axis tick label."""
    head = str(title).split(":")[0].strip()
    # Cut at the subtitle only when the main title can stand on its own: "Public
    # Goods" does not name Ledyard's survey, but "Governing the Commons" does.
    short = head if len(head) >= 20 else str(title).strip()
    stamp = str(canonical_surname.get(first_author.get(work_id, ""), "") or "")
    if pd.notna(year) and int(year) > 0:
        stamp = f"{stamp} {int(year)}".strip()
    lines = textwrap.wrap(short.rstrip("."), width=width) or [""]
    if len(lines) > 2:
        lines = lines[:2]
        lines[1] = lines[1].rstrip(",;:") + "…"
    if stamp:
        lines[-1] += f" ({stamp})"
    return "\n".join(lines)


# Full ranking for the long tail; the figure shows only the head.
canon_table = (
    canon_graph.top_cited(canon_graph.n_works)
    .reset_index()
    .assign(First_author=lambda d: d["id"].map(first_author).map(canonical_surname))
    .loc[:, ["id", "citation_count", "First_author", "Year", "Title", "Journal_Canonical", "ring"]]
    .rename(columns={"citation_count": "Citing_publications", "ring": "Ring"})
)
canon_table_csv = FIGURE_DIR / "most_cited_works.csv"
canon_table.to_csv(canon_table_csv, index=False)
display(canon_table.head(10))

# Ascending order puts the most-cited work at the top of a horizontal bar chart.
plot_works = canon_works.iloc[::-1]
is_core = plot_works["ring"].eq(0).to_numpy()
n_core_in_top = int(is_core.sum())

with econ_style():
    fig, ax = plt.subplots(figsize=figure_size("text", ratio=0.82))

    work_positions = list(range(len(plot_works)))
    canon_bars = barh(
        ax,
        work_positions,
        plot_works["citation_count"],
        height=0.72,
        color=[GRAYS[3] if core else GRAYS[1] for core in is_core],
        hatch=[HATCHES[1] if core else "" for core in is_core],
    )
    value_labels(ax, canon_bars, padding=3)
    ax.set_yticks(work_positions)
    ax.set_yticklabels(
        [
            work_label(work_id, title, year)
            for work_id, title, year in zip(
                plot_works.index, plot_works["Title"], plot_works["Year"], strict=True
            )
        ]
    )
    ax.tick_params(axis="y", length=0)
    ax.set_xlabel("Number of publications citing the work")
    integer_ticks(ax, "x")
    ax.set_xlim(0, plot_works["citation_count"].max() * 1.15)
    ax.margins(y=0.02)
    ax.legend(
        handles=[
            Patch(
                facecolor=GRAYS[1],
                edgecolor="black",
                linewidth=0.6,
                label="From the wider literature",
            ),
            Patch(
                facecolor=GRAYS[3],
                edgecolor="black",
                linewidth=0.6,
                hatch=HATCHES[1],
                label=f"One of the {canon_graph.n_core_works} core publications",
            ),
        ],
        frameon=False,
        loc="lower right",
        fontsize=7.5,
        handlelength=1.4,
        borderaxespad=0.4,
    )

    canon_caption = "The corpus's shared canon"
    canon_notes = (
        f"The {TOP_WORKS} works cited by the largest number of the {canon_graph.n_core_works} "
        f"core publications, counting each publication once per cited work however many times "
        f"its bibliography lists it. Counts use the citation graph's canonical works, so "
        f"citation variants of one work are already merged. {n_core_in_top} of the "
        f"{TOP_WORKS} are themselves core publications (hatched): the corpus cites its own "
        f"authors alongside the wider literature. Titles are cut at the subtitle where the "
        f"main title stands alone; the first author and year identify the work. The full "
        f"ranking of all {len(canon_table):,} cited works is saved beside this figure as "
        f"most_cited_works.csv."
    )
    canon_files = save_figure(
        fig,
        FIGURE_DIR / "canon_most_cited_works",
        caption=canon_caption,
        notes=canon_notes,
        source=SOURCE_NOTE,
    )
    plt.show()

show_caption(canon_caption, notes=canon_notes)
print(f"Saved: {canon_table_csv}")
for path in canon_files:
    print(f"Saved: {path}")

## Appendix. Whom the focal author is read alongside

The ego network below is descriptive. It repeats, drawn as a star, what the citation
ranking in section 5 already says: the authors most often co-cited with the focal author
are the most-cited authors in the corpus. It is kept for readers who want to see the raw
tie weights behind the map, and belongs in an appendix rather than the main text.

In [ ]:
EGO_PARTNERS = 15
focus_ties = sorted(
    cocitation_network[focus_author_id].items(),
    key=lambda item: item[1]["weight"],
    reverse=True,
)
ego_partners = [author_id for author_id, _ in focus_ties[:EGO_PARTNERS]]
# `subgraph()` keeps its node filter in a set, so the copy's node order follows
# string hashing and varies with PYTHONHASHSEED. `spring_layout` seeds initial
# positions by node index, so the fixed seed below started from a different
# configuration on every run and this figure never redrew the same twice.
# Rebuild in sorted node order -- the convention author_cocitation_network()
# already uses -- so the seed reproduces.
_ego_view = cocitation_network.subgraph([focus_author_id, *ego_partners])
ego_network = nx.Graph()
ego_network.add_nodes_from(sorted(_ego_view.nodes()))
# Edge order is hash-ordered too, and it sets which tie is painted over which,
# so normalise orientation and sort before inserting.
ego_network.add_edges_from(
    sorted((u, v, d) if u <= v else (v, u, d) for u, v, d in _ego_view.edges(data=True))
)
ego_min_weight = focus_ties[EGO_PARTNERS - 1][1]["weight"]

ego_positions = nx.spring_layout(ego_network, weight="weight", seed=3, k=1.4, iterations=400)

with econ_style():
    fig, ax = plt.subplots(figsize=figure_size("text", ratio=0.78))

    ego_node_sizes = {a: 300 if a == focus_author_id else 95 for a in ego_network}
    ego_edges = [(u, v) for u, v in ego_network.edges() if focus_author_id in (u, v)]
    alter_edges = [(u, v) for u, v in ego_network.edges() if focus_author_id not in (u, v)]
    max_ego_weight = max(cocitation_network[focus_author_id][a]["weight"] for a in ego_partners)

    nx.draw_networkx_edges(
        ego_network,
        ego_positions,
        ax=ax,
        edgelist=alter_edges,
        width=0.4,
        edge_color=EDGE_COLOR,
        alpha=0.7,
    )
    nx.draw_networkx_edges(
        ego_network,
        ego_positions,
        ax=ax,
        edgelist=ego_edges,
        width=[0.5 + 2.4 * (ego_network[u][v]["weight"] / max_ego_weight) for u, v in ego_edges],
        edge_color="black",
        alpha=0.85,
    )
    nx.draw_networkx_nodes(
        ego_network,
        ego_positions,
        ax=ax,
        node_size=[ego_node_sizes[a] for a in ego_network],
        node_color=["black" if a == focus_author_id else NODE_FILL for a in ego_network],
        edgecolors="black",
        linewidths=0.6,
    )
    draw_node_labels(ax, ego_positions, ego_node_sizes, focus_author_id, fontsize=7)
    network_axes(ax, equal=False)
    ax.margins(0.13)

    ego_caption = f"Whom {focus_author_name} is cited alongside"
    ego_notes = (
        f"The {EGO_PARTNERS} authors most often co-cited with {focus_author_name} in the author "
        f"co-citation network. Heavy tie width is proportional to the number of source "
        f"publications citing {focus_author_name} and that author together, ranging from "
        f"{ego_min_weight} to {max_ego_weight} publications. Light ties among the other authors "
        f"show that they form one connected research area rather than separate groups. Positions "
        f"come from a force-directed layout with a fixed seed and carry no units."
    )
    ego_files = save_figure(
        fig,
        FIGURE_DIR / "cardenas_cocitation_ego_network",
        caption=ego_caption,
        notes=ego_notes,
        source=SOURCE_NOTE,
    )
    plt.show()

show_caption(ego_caption, notes=ego_notes)
for path in ego_files:
    print(f"Saved: {path}")